# BRIDGE revision 2 — Part C1 — leave-one-dataset-out: TCH-Net, H-only, 5 neural baselines and RF/XGBoost, 5 seeds, purged calibration, labelling-budget curves.

Settings: **GPU T4 x1** (or P100), Internet off, the same four datasets attached as before.
Run with *Save Version → Save & Run All*. Results are saved incrementally to
`tch_net_v2_results/partC1_results.json`, so a run stopped by the 12-hour limit still keeps
everything finished before the stop. Download the whole `tch_net_v2_results` folder at the end.

What changed in revision 2 (all parts share this setup): exact de-duplication before splitting,
CICIDS-2017 `_plus` duplicate files skipped, explicit unit conversions (UNSW-NB15), strict
feature mappings, derived rate/total slots, odd window W = 33 (no label ties), non-overlapping
evaluation windows, windows described as random record bags (rows are shuffled at load).

In [1]:
import os, glob, json, time, warnings, copy, gc, itertools, math
warnings.filterwarnings('ignore')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython()
    import matplotlib.pyplot as plt
except NameError:
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import matplotlib.ticker as ticker

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    f1_score, accuracy_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, auc, matthews_corrcoef)
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from scipy import stats as scipy_stats

try:
    from xgboost import XGBClassifier; XGB_OK = True
except ImportError:
    XGB_OK = False; print("XGBoost unavailable — fallback to GradientBoosting")

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True      # fastest CUDA kernels
    torch.backends.cudnn.deterministic = False  # benchmark requires non-deterministic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")
print(f"CUDA: {torch.cuda.is_available()} | XGBoost: {XGB_OK}")


PyTorch 2.11.0+cu128 | Device: cuda
CUDA: True | XGBoost: True


## Canonical Feature Vocabulary (46 features)

These are **real network-flow features** with fixed physical meaning.
Every feature keeps its name and units regardless of which dataset it came from.
Features not present in a dataset are zero-filled — never fabricated.

**Zero-fill policy (known limitation):** Absent features are set to 0, which conflates
"true value is zero" with "feature not observed in this dataset." This is a design choice,
not an oversight. Mitigation: `combine()._balance()` enforces strict 1:1 class balance
*within* each dataset block before concatenation, removing the main shortcut-learning risk
(predicting label from dataset identity via the missingness fingerprint). Cell 7b quantifies
this directly — a missingness-only classifier achieves accuracy indistinguishable from the
majority-class baseline. Post-scaling behaviour of zero-filled slots is also characterised
in Cell 7b.

**Grouping for T/C/H branches:**
- **T-branch temporal** (indices 0-16): rates, durations, counts that change over time
- **H-branch statistical** (indices 17-37): packet size distributions, IAT distributions
- **Both branches** (indices 38-45): TCP flags, header info, window sizes

**Architecture citations for code below:**
- Depthwise Separable Convolutions: Howard et al. (2017) MobileNets
- Squeeze-Excitation blocks: Hu et al. (2018) SE-Net, CVPR
- Pre-LayerNorm stability: Xiong et al. (2020), ICML


In [2]:
# ── 46 canonical features ─────────────────────────────────────────────────
SEMANTIC_FEATURES = [
    # ── Flow-level counts & rates (T-branch primary) ─────────────────────
    "flow_duration",        # 0  total flow duration
    "pkt_count_fwd",        # 1  total forward packets
    "pkt_count_bwd",        # 2  total backward packets
    "byte_count_fwd",       # 3  total forward bytes
    "byte_count_bwd",       # 4  total backward bytes
    "pkt_rate",             # 5  packets per second (flow level)
    "byte_rate",            # 6  bytes per second
    "fwd_pkt_rate",         # 7  forward packets per second
    "bwd_pkt_rate",         # 8  backward packets per second
    "fwd_byte_rate",        # 9  forward bytes per second
    "bwd_byte_rate",        # 10 backward bytes per second
    "pkt_count_total",      # 11 total packets (fwd + bwd)
    "byte_count_total",     # 12 total bytes
    "fwd_pkt_len_total",    # 13 total length of fwd packets
    "bwd_pkt_len_total",    # 14 total length of bwd packets
    "subflow_fwd_pkts",     # 15 subflow forward packets
    "subflow_bwd_pkts",     # 16 subflow backward packets
    # ── Packet size statistics (H-branch primary) ────────────────────────
    "pkt_len_min",          # 17
    "pkt_len_max",          # 18
    "pkt_len_mean",         # 19
    "pkt_len_std",          # 20
    "pkt_len_var",          # 21
    "fwd_pkt_len_min",      # 22
    "fwd_pkt_len_max",      # 23
    "fwd_pkt_len_mean",     # 24
    "fwd_pkt_len_std",      # 25
    "bwd_pkt_len_min",      # 26
    "bwd_pkt_len_max",      # 27
    "bwd_pkt_len_mean",     # 28
    "bwd_pkt_len_std",      # 29
    # ── IAT statistics (T + H) ───────────────────────────────────────────
    "iat_mean",             # 30
    "iat_std",              # 31
    "iat_max",              # 32
    "iat_min",              # 33
    "fwd_iat_mean",         # 34
    "fwd_iat_std",          # 35
    "bwd_iat_mean",         # 36
    "bwd_iat_std",          # 37
    # ── TCP flags (H-branch) ─────────────────────────────────────────────
    "flag_syn",             # 38
    "flag_ack",             # 39
    "flag_fin",             # 40
    "flag_rst",             # 41
    "flag_psh",             # 42
    "flag_urg",             # 43
    # ── Header / window ──────────────────────────────────────────────────
    "fwd_header_len",       # 44
    "init_win_fwd",         # 45
]
N_SEM = len(SEMANTIC_FEATURES)
SEM_IDX = {name: i for i, name in enumerate(SEMANTIC_FEATURES)}
print(f"Canonical vocabulary: {N_SEM} features")

# ── Per-dataset alias maps ───────────────────────────────────────────────
# Key = canonical name. Value = list of possible column names (case-insensitive).
# ONLY genuinely equivalent features are mapped. No fabricated proxies.
# Short aliases (<5 chars) NEVER matched via substring — only exact match.
#
# ── DATASET RESET (v7) ────────────────────────────────────────────────────
# CIC-IoT-DIAD-2024 (Mostafa Ali Ogab, Kaggle) was confirmed BROKEN — every
# file (including attack-named files) carries "NeedManualLabel" in the Label
# column. Genuinely unusable for supervised learning. DROPPED.
# CICIoMT-2024 (Lian Mollick, Kaggle) was confirmed WRONG VARIANT — 45 cols,
# no label column at all (same reduced schema as CIC-IoT-2023). DROPPED.
#
# Replaced with, and verified directly against user-supplied sample CSVs:
#   • CICIDS-2017                          (sample: cicids-2017-friday[-plus].csv)
#   • CIC-BCCC-NRC-TabularIoTAttacks-2024   (sample: cic-bccc-nrc-tabulariotAttack.csv)
#   • CICIoMT-2024 (Train-Test, D. Kilichev) (sample: ciciomt-{train,test}-set.csv)

# ── ALIAS_FLOWMETER ──────────────────────────────────────────────────────
# Shared by CICIDS-2017 AND CIC-BCCC-NRC-TabularIoTAttacks-2024.
# Verified column-for-column against both uploaded samples — they use the
# IDENTICAL CICFlowMeter naming convention (BCCC's updated extractor):
#   "Total Fwd Packet"           (singular, no trailing 's')
#   "Total Bwd packets"          (lowercase 'p', trailing 's')
#   "Total Length of Fwd Packet" (singular)
#   "Total Length of Bwd Packet" (singular)
#   "FWD Init Win Bytes"         (upper-case FWD)
# Old verbose CICIDS MachineLearningCSV.zip names ("Total Fwd Packets",
# "Init_Win_bytes_forward", ...) are kept as fallback aliases in case a
# differently-formatted CICIDS-2017 file is substituted later.
# cic-bccc-nrc-tabulariotAttack.csv additionally carries an explicit
# "Attack Name" string column (e.g. "Benign Traffic") used for labeling —
# see load_cic_bccc_nrc() below — it is NOT part of the feature vocabulary.
ALIAS_FLOWMETER = {
    "flow_duration":     ["Flow Duration"],
    "pkt_count_fwd":     ["Total Fwd Packet",              "Total Fwd Packets",
                          "Tot Fwd Pkts"],
    "pkt_count_bwd":     ["Total Bwd packets",             "Total Backward Packets",
                          "Tot Bwd Pkts"],
    "byte_count_fwd":    ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "byte_count_bwd":    ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "pkt_rate":          ["Flow Packets/s",                "Flow Pkts/s"],
    "byte_rate":         ["Flow Bytes/s",                  "Flow Byts/s"],
    "fwd_pkt_rate":      ["Fwd Packets/s",                 "Fwd Pkts/s"],
    "bwd_pkt_rate":      ["Bwd Packets/s",                 "Bwd Pkts/s"],
    "fwd_pkt_len_total": ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "bwd_pkt_len_total": ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "subflow_fwd_pkts":  ["Subflow Fwd Packets",           "Subflow Fwd Pkts"],
    "subflow_bwd_pkts":  ["Subflow Bwd Packets",           "Subflow Bwd Pkts"],
    "pkt_len_min":       ["Packet Length Min",             "Min Packet Length",
                          "Pkt Len Min"],
    "pkt_len_max":       ["Packet Length Max",             "Max Packet Length",
                          "Pkt Len Max"],
    "pkt_len_mean":      ["Packet Length Mean",            "Pkt Len Mean"],
    "pkt_len_std":       ["Packet Length Std",             "Pkt Len Std"],
    "pkt_len_var":       ["Packet Length Variance",        "Pkt Len Var"],
    "fwd_pkt_len_min":   ["Fwd Packet Length Min",         "Fwd Pkt Len Min"],
    "fwd_pkt_len_max":   ["Fwd Packet Length Max",         "Fwd Pkt Len Max"],
    "fwd_pkt_len_mean":  ["Fwd Packet Length Mean",        "Fwd Pkt Len Mean"],
    "fwd_pkt_len_std":   ["Fwd Packet Length Std",         "Fwd Pkt Len Std"],
    "bwd_pkt_len_min":   ["Bwd Packet Length Min",         "Bwd Pkt Len Min"],
    "bwd_pkt_len_max":   ["Bwd Packet Length Max",         "Bwd Pkt Len Max"],
    "bwd_pkt_len_mean":  ["Bwd Packet Length Mean",        "Bwd Pkt Len Mean"],
    "bwd_pkt_len_std":   ["Bwd Packet Length Std",         "Bwd Pkt Len Std"],
    "iat_mean":          ["Flow IAT Mean"],
    "iat_std":           ["Flow IAT Std"],
    "iat_max":           ["Flow IAT Max"],
    "iat_min":           ["Flow IAT Min"],
    "fwd_iat_mean":      ["Fwd IAT Mean"],
    "fwd_iat_std":       ["Fwd IAT Std"],
    "bwd_iat_mean":      ["Bwd IAT Mean"],
    "bwd_iat_std":       ["Bwd IAT Std"],
    "flag_syn":          ["SYN Flag Count",               "SYN Flag Cnt"],
    "flag_ack":          ["ACK Flag Count",               "ACK Flag Cnt"],
    "flag_fin":          ["FIN Flag Count",               "FIN Flag Cnt"],
    "flag_rst":          ["RST Flag Count",               "RST Flag Cnt"],
    "flag_psh":          ["PSH Flag Count",               "PSH Flag Cnt"],
    "flag_urg":          ["URG Flag Count",               "URG Flag Cnt"],
    "fwd_header_len":    ["Fwd Header Length",            "Fwd Header Len"],
    "init_win_fwd":      ["FWD Init Win Bytes",           "Init_Win_bytes_forward",
                          "Init Fwd Win Bytes",           "Init Fwd Win Byts"],
}
# Coverage: 42/46 (91%) — verified via live smoke test directly against both uploaded sample CSVs
# (cicids-2017-friday.csv, cic-bccc-nrc-tabulariotAttack.csv).
# 4 columns present in the CICIDS-2017 sample only (ICMP Code, ICMP Type,
# Total TCP Flow Time, Attempted Category) and 1 present in the BCCC sample
# only (Attack Name) are NOT canonical features and are intentionally unmapped.
# Unmapped canonical features (8): fwd_byte_rate, bwd_byte_rate (no per-direction
# byte-rate column in either dataset), pkt_count_total / byte_count_total
# (derived in combine() from fwd+bwd, not read from a column).

# ── ALIAS_IOMT ───────────────────────────────────────────────────────────
# CICIoMT-2024 (Train-Test split, D. Kilichev re-upload). 46 columns.
# Verified against: ciciomt-train-set.csv, ciciomt-test-set.csv
# SAME 45-feature schema as the a known-broken re-upload variant, but THIS
# variant carries a genuine, populated "Label" column:
#   train sample Label values: "DoS-TCP"
#   test  sample Label values: "Malformed_Data"
# (Dadkhah et al. 2024 — 19 attack-class string labels + benign; benign
# rows are NOT present in either 10-row sample, so the exact benign string
# is unconfirmed — load_ciciomt() below matches ANY label containing
# "benign" case-insensitively, not just an exact string, to stay robust
# to "Benign" / "BenignTraffic" / "Benign Traffic" variants.)
# Coverage: 17/46 (37%) features ALIAS-RESOLVE correctly after the Srate/Drate
# correction below. The EFFECTIVE coverage, measured empirically as "has at
# least one non-zero value across the full loaded dataset" (the stricter,
# more defensible metric — see Cell 6's feature_coverage_report.csv), is
# 16/46 (35%): "Drate" (-> bwd_pkt_rate) resolves to a real column, but that
# column's values are ~0 across virtually every row in this dataset — which
# independently matches the original CICIoT2023 paper's own published stats
# (Drate mean=5.46e-6, median/25/50/75th percentiles all exactly 0). This is
# a genuine property of the data, not a mapping defect. Separately, "Srate"
# (-> fwd_pkt_rate) and "Rate" (-> pkt_rate) were found to be numerically
# IDENTICAL in this dataset (again matching the source paper's stats table
# exactly) — both alias-resolve and both carry signal, but they are
# redundant with each other, not independent features. Report 16/46 (35%)
# as the dataset's effective coverage in the paper, not 17/46 — it is the
# number that will reproduce if a reviewer re-runs this pipeline.
# Protocol-presence flags (HTTP, HTTPS, TCP, UDP etc.) and statistical
# moment features (Magnitue, Radius, Covariance, Weight) have no
# CICFlowMeter equivalents — correctly zero-filled.
ALIAS_IOMT = {
    # REVISION 2 (R3 §3.2, R4-1): STRICT mapping. A slot is mapped only when the
    # CICIoMT-2024 column measures the same quantity in a convertible unit.
    # Audit evidence (BRIDGE_data_audit.ipynb) for every removal is in UNIT_TABLE.
    "pkt_count_total":  ["Number", "number"],
    "byte_count_total": ["Tot sum",  "tot sum"],
    "pkt_rate":         ["Rate",     "rate"],
    "pkt_len_min":      ["Min",      "min"],
    "pkt_len_max":      ["Max",      "max"],
    "pkt_len_mean":     ["AVG",      "avg"],
    "pkt_len_std":      ["Std",      "std"],
    "pkt_len_var":      ["Variance", "variance"],
    # flag COUNTS (not the 0/1 "*_flag_number" indicators, which are proportions)
    "flag_syn":         ["syn_count"],
    "flag_ack":         ["ack_count"],
    "flag_fin":         ["fin_count"],
    "flag_rst":         ["rst_count"],
    # REMOVED in revision 2:
    #  Srate -> fwd_pkt_rate : numerically identical to Rate, so not a forward-only rate
    #  Drate -> bwd_pkt_rate : ~0 on every row; not a usable backward rate
    #  IAT   -> iat_mean     : median 8.5e7, p99 1.7e8 — behaves like a timestamp, not an interval
    #  Header_Length -> fwd_header_len : scale (p99 3.4e5) incompatible with per-flow header bytes
    #  psh_flag_number -> flag_psh : only a 0/1 proportion exists, no count
}
# Srate / Drate added after cross-checking against the CICIoT2023 extractor's
# own published feature definitions (Neto et al. 2023, Sensors 23(13):5941;
# corroborated by multiple independent secondary sources): "Srate = rate of
# outbound packets' transmission in a flow", "Drate = rate of inbound
# packets' transmission in a flow" — directly equivalent to fwd_pkt_rate /
# bwd_pkt_rate (forward/backward packet rate), just computed over this
# extractor's sliding-window unit instead of a CICFlowMeter 5-tuple flow.
#
# Explicitly REJECTED: mapping the "Duration" column to "flow_duration".
# It looks tempting from the name alone, but the same paper's feature table
# documents "Duration" as the packet's IP Time-To-Live (TTL) — confirmed by
# its value range (0-255, matching an 8-bit TTL field) in the paper's
# descriptive statistics table. Mapping it to flow_duration would be a
# genuine semantic error (different physical quantity), not a coverage gain.
# This dataset's actual "flow_duration" column (present in the official
# 47-feature CICIoT2023 schema) is simply ABSENT from this 46-col CICIoMT2024
# re-upload — a genuine schema gap, not an alias-mapping oversight.
#
# Also explicitly NOT mapped (genuine schema mismatch, verified against the
# same source): Protocol Type (categorical, not a canonical feature),
# ece_flag_number / cwr_flag_number (no ECE/CWR slots in the 46-feature
# vocabulary), all protocol-presence one-hot flags (HTTP/HTTPS/DNS/.../LLC —
# no protocol-indicator slot exists in the vocabulary), and Magnitude /
# Radius / Covariance / Weight (composite cross-direction statistics with
# no single-feature canonical equivalent — e.g. Weight is explicitly defined
# as incoming_packets x outgoing_packets, a multiplicative composite that
# does not correspond to any additive canonical count we have).
# Coverage after this correction: 17/46 (37%) — up from 15/46, verified by
# re-running build_semantic_vector() against the real sample CSVs.
# NOTE: Short aliases (Min, Max, AVG, Std, IAT, Rate) are length < 5 chars.
# They are correctly handled by the exact case-insensitive match in
# build_semantic_vector() step 2, which runs BEFORE the substring guard.
# The substring guard in step 3 (len < 5 → skip) only applies to substring
# fallback — exact alias match is always attempted first and has no length guard.
# Coverage: 15/46 (genuine ceiling — extractor design limitation)

# ── ALIAS_UNSW ────────────────────────────────────────────────────────────
# UNSW-NB15 (Moustafa & Slay, MilCIS 2015). Argus + Bro/Zeek extractor — a
# THIRD, genuinely distinct extractor family (neither CICFlowMeter nor the
# CIC-IoT sliding-window statistical extractor). This is the heterogeneity
# upgrade: BRIDGE now spans three independent flow-extraction toolchains.
# Enterprise/general network testbed (NOT IoT-specific — same scope caveat as
# CICIDS-2017; see UNSW_SCOPE_NOTE below).
#
# Verified against the CLEAN labelled split shipped by the Kaggle mirror
# (mrwellsdavid/unsw-nb15): UNSW_NB15_training-set.csv / _testing-set.csv —
# 45 columns WITH a header, binary numeric `label` (0=normal, 1=attack) and a
# human-readable `attack_cat` family string. load_unsw_nb15() reads ONLY files
# that expose BOTH a `label` and a `dur` header column, which automatically
# rejects the raw headerless UNSW-NB15_1..4.csv captures and the
# NUSW-NB15_features.csv metadata file.
#
# Mappings follow the ORIGINAL 2015 feature definitions (Moustafa & Slay),
# mapped by PHYSICAL quantity, not by name:
#   dur    = record total duration                    -> flow_duration
#   spkts  = source->dest packet count                -> pkt_count_fwd
#   dpkts  = dest->source packet count                -> pkt_count_bwd
#   sbytes = source->dest transaction bytes           -> byte_count_fwd
#                                                        + fwd_pkt_len_total
#   dbytes = dest->source transaction bytes           -> byte_count_bwd
#                                                        + bwd_pkt_len_total
#   smean  = mean packet size, source direction       -> fwd_pkt_len_mean
#   dmean  = mean packet size, dest direction         -> bwd_pkt_len_mean
#   sinpkt = source inter-packet arrival time (mSec)  -> fwd_iat_mean
#   dinpkt = dest inter-packet arrival time (mSec)    -> bwd_iat_mean
#   sload  = source data rate (BITS/sec)              -> fwd_byte_rate  [A]
#   dload  = dest data rate (BITS/sec)                -> bwd_byte_rate  [A]
#   swin   = source TCP window advertisement value    -> init_win_fwd   [B]
#
# [A] sload/dload are BITS/sec; fwd_byte_rate/bwd_byte_rate are BYTES/sec — a
#     constant 8x unit factor fully absorbed by the per-feature RobustScaler
#     (scale-invariant), so the mapping is honest at the level of
#     "forward/backward throughput". The unit difference is noted, not hidden.
# [B] swin is the advertised TCP receive-window (source side); init_win_fwd
#     (CICFlowMeter) is the initial forward-window byte count. Both encode the
#     forward TCP window size; mapped with this caveat noted.
#
# EXPLICITLY REJECTED (kept unmapped on purpose — Section 3.4.4 discipline):
#   * rate -> pkt_rate : the derived `rate` column's exact physical definition
#     is NOT in Moustafa & Slay's canonical 49-feature table (added only in the
#     derived train/test split). Rather than guess it is the flow packet-rate,
#     it is left unmapped. (Loss: 1 slot.)
#   * sjit/djit -> iat_std : jitter (mean deviation of packet delay) and the
#     std of inter-arrival times are correlated but NOT the same statistic. No
#     canonical "jitter" slot exists; mapping to iat_std would be a semantic
#     approximation, so it is rejected. (Loss: 2 slots.)
#   * synack/ackdat -> flag_* : these are TCP setup *times* (SYN->SYN_ACK,
#     SYN_ACK->ACK), NOT flag counts. UNSW-NB15's clean split carries NO TCP
#     flag-count columns (only a categorical `state` and these setup times), so
#     ALL flag_* slots are correctly left unmapped — a genuine schema gap, not
#     an oversight. This differing coverage profile vs the CICFlowMeter datasets
#     is exactly the point of a heterogeneous benchmark.
#   * sttl/dttl (IP TTL), stcpb/dtcpb (TCP base seq), tcprtt, sloss/dloss, dwin,
#     proto/service/state (categorical), ct_* (connection-count aggregates),
#     trans_depth/response_body_len (HTTP), is_* : no canonical flow-statistic
#     equivalent — correctly zero-filled.
#
# Effective coverage: 14/46 canonical slots resolve (12 distinct source
# columns; sbytes/dbytes each fill two slots, exactly as ALIAS_FLOWMETER
# dual-maps byte_count_* and *_pkt_len_total). pkt_count_total is additionally
# derived in combine() from fwd+bwd (not read from a column), same as the other
# datasets. Confirm the live number via Cell 6's coverage table and the Cell 2b
# diagnostic after attaching the real file.
ALIAS_UNSW = {
    # REVISION 2 (R3 §3.2, R4-1): units are converted explicitly in
    # apply_units_and_derivations() — never left to the scaler.
    "flow_duration":     ["dur"],       # seconds -> converted to microseconds
    "pkt_count_fwd":     ["spkts"],
    "pkt_count_bwd":     ["dpkts"],
    "byte_count_fwd":    ["sbytes"],
    "byte_count_bwd":    ["dbytes"],
    "fwd_pkt_len_total": ["sbytes"],
    "bwd_pkt_len_total": ["dbytes"],
    "fwd_pkt_len_mean":  ["smean"],
    "bwd_pkt_len_mean":  ["dmean"],
    "fwd_iat_mean":      ["sinpkt"],    # milliseconds -> converted to microseconds
    "bwd_iat_mean":      ["dinpkt"],    # milliseconds -> converted to microseconds
    # REMOVED in revision 2:
    #  sload/dload -> fwd/bwd_byte_rate : bits/s with an Argus-specific basis (audit:
    #      sload/(sbytes/dur) median 6, not 8). The canonical bytes/s rates are now
    #      DERIVED as sbytes/dur and dbytes/dur, the same definition as CICFlowMeter.
    #  swin -> init_win_fwd : values are 0 or 255 only; not the initial window in bytes.
}

ALIAS_MAPS = {
    "CICIDS-2017":                          ALIAS_FLOWMETER,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  ALIAS_FLOWMETER,
    "CICIoMT-2024":                         ALIAS_IOMT,
    "UNSW-NB15":                             ALIAS_UNSW,
}

def build_semantic_vector(df, ds_name):
    alias_map = ALIAS_MAPS.get(ds_name, {})
    col_lower = {c.strip().lower(): c for c in df.columns}
    out = np.zeros((len(df), N_SEM), dtype=np.float32)
    matched = []
    for sem_name in SEMANTIC_FEATURES:
        si = SEM_IDX[sem_name]
        col = None
        # 1. Exact match (case-insensitive)
        if sem_name in col_lower:
            col = col_lower[sem_name]
        # 2. Alias match (exact, case-insensitive — no length guard here)
        elif sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if al in col_lower:
                    col = col_lower[al]; break
        # 3. Substring fallback — MINIMUM LENGTH GUARD (len < 5 → skip)
        # Short aliases like "Min", "Max", "Std", "IAT", "Rate" must NOT
        # match via substring — prevents false hits on unrelated columns.
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if len(al) < 5:
                    continue
                ms = [c for c in df.columns if al in c.strip().lower()]
                if ms: col = ms[0]; break
        # 3b. Normalised substring fallback (strip spaces/underscores)
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower().replace(' ','').replace('_','')
                if len(al) < 5:
                    continue
                for c in df.columns:
                    cn = c.strip().lower().replace(' ','').replace('_','')
                    if al == cn or (len(al) > 4 and al in cn):
                        col = c; break
                if col: break
        if col is not None:
            vals = pd.to_numeric(df[col], errors='coerce').fillna(0).values
            out[:, si] = vals.astype(np.float32)
            matched.append(sem_name)
    return out, matched



# ═════════════════════════════════════════════════════════════════════════════
# REVISION 2 — explicit units, derived slots, de-duplication (R3 §3.2, §3.7; R4-1)
# ═════════════════════════════════════════════════════════════════════════════
# Canonical units: durations and inter-arrival times in MICROSECONDS, rates per
# SECOND, sizes in BYTES, counts per flow. Every conversion is executable code
# below; nothing is left to the RobustScaler.
UNIT_TABLE = {
    'CICIDS-2017': {
        'flow_duration / iat_*': 'us (CICFlowMeter); audit: bytes/(Flow Bytes/s * duration) = 1e-6, duration = iat_mean*(n-1)',
        'rates': 'per second (native)', 'sizes': 'bytes', 'conversion': 'none'},
    'CIC-BCCC-NRC-TabularIoTAttacks-2024': {
        'flow_duration / iat_*': 'us (CICFlowMeter-compatible); same audit checks as CICIDS-2017 pass',
        'rates': 'per second (native)', 'sizes': 'bytes', 'conversion': 'none'},
    'CICIoMT-2024': {
        'rates': 'Rate = packets/s over the extractor\'s packet window',
        'sizes': 'bytes, window-averaged', 'flags': '*_count = per-flow counts, window-averaged',
        'conversion': 'none; IAT, Srate, Drate, Header_Length, psh_flag_number unmapped (see ALIAS_IOMT)'},
    'UNSW-NB15': {
        'flow_duration': 'dur seconds -> x 1e6 (us); audit: sinpkt*(spkts-1)/dur = 1000',
        'fwd/bwd_iat_mean': 'sinpkt/dinpkt milliseconds -> x 1e3 (us)',
        'byte rates': 'derived sbytes/dur, dbytes/dur (bytes/s); sload/dload unmapped',
        'conversion': 'dur x1e6, sinpkt/dinpkt x1e3'},
}
_DUR_TO_US = {'UNSW-NB15': 1e6}
_IAT_TO_US = {'UNSW-NB15': 1e3}
# Slots that are computable from other slots with the canonical definition.
_DERIVATIONS = [
    ('pkt_count_total',  'sum',  ('pkt_count_fwd', 'pkt_count_bwd')),
    ('byte_count_total', 'sum',  ('byte_count_fwd', 'byte_count_bwd')),
    ('pkt_rate',         'rate', ('pkt_count_total',)),
    ('byte_rate',        'rate', ('byte_count_total',)),
    ('fwd_pkt_rate',     'rate', ('pkt_count_fwd',)),
    ('bwd_pkt_rate',     'rate', ('pkt_count_bwd',)),
    ('fwd_byte_rate',    'rate', ('byte_count_fwd',)),
    ('bwd_byte_rate',    'rate', ('byte_count_bwd',)),
]
DERIVED_SLOTS = {}

def apply_units_and_derivations(X, ds_name):
    """Convert to canonical units, then fill slots that are NOT mapped natively for
       this dataset but are computable from mapped ones (schema-level decision, so
       every file of a dataset gets the same slots)."""
    X = X.astype(np.float64)
    I = SEM_IDX
    if ds_name in _DUR_TO_US:
        X[:, I['flow_duration']] *= _DUR_TO_US[ds_name]
    if ds_name in _IAT_TO_US:
        for f in ('fwd_iat_mean', 'bwd_iat_mean', 'iat_mean', 'iat_std', 'iat_max', 'iat_min',
                  'fwd_iat_std', 'bwd_iat_std'):
            if f in ALIAS_MAPS.get(ds_name, {}): X[:, I[f]] *= _IAT_TO_US[ds_name]
    present = set(ALIAS_MAPS.get(ds_name, {}).keys())
    derived = []
    dur_s = X[:, I['flow_duration']] / 1e6 if 'flow_duration' in present else None
    for slot, kind, srcs in _DERIVATIONS:
        if slot in present or not all(s in present for s in srcs):
            continue
        if kind == 'sum':
            X[:, I[slot]] = X[:, I[srcs[0]]] + X[:, I[srcs[1]]]
        else:
            if dur_s is None: continue
            ok = dur_s > 0
            X[:, I[slot]] = 0.0
            X[ok, I[slot]] = X[ok, I[srcs[0]]] / dur_s[ok]
        present.add(slot); derived.append(slot)
    DERIVED_SLOTS[ds_name] = derived
    return X.astype(np.float32)

DEDUP_STATS = {}
def dedup_canonical(X, y, ds_name, extras=()):
    """Drop exact duplicate canonical vectors (keep first), and drop every copy of a
       vector that occurs with BOTH labels. Applied per dataset BEFORE balancing and
       splitting, so no vector can occur in two partitions (R3 §3.7)."""
    n0 = len(y)
    Xc = np.ascontiguousarray(X.astype(np.float32) + np.float32(0.0))   # -0.0 -> +0.0
    key = Xc.view(np.dtype((np.void, Xc.dtype.itemsize * Xc.shape[1]))).ravel()
    _, first, inv = np.unique(key, return_index=True, return_inverse=True)
    inv = inv.ravel()
    lo = np.full(len(first), 2, dtype=np.int64); hi = np.full(len(first), -1, dtype=np.int64)
    np.minimum.at(lo, inv, y.astype(np.int64)); np.maximum.at(hi, inv, y.astype(np.int64))
    conflict = lo != hi
    keep = np.sort(first[~conflict])
    DEDUP_STATS[ds_name] = {'rows_in': int(n0), 'unique_vectors': int(len(first)),
                            'conflicting_vectors_dropped': int(conflict.sum()),
                            'rows_kept': int(len(keep)),
                            'duplicate_frac': float(1 - len(first) / max(n0, 1))}
    print(f"  De-duplication: {n0:,} rows -> {len(keep):,} unique, label-consistent vectors "
          f"({DEDUP_STATS[ds_name]['duplicate_frac']*100:.1f}% duplicates, "
          f"{int(conflict.sum()):,} conflicting vectors dropped)")
    return (X[keep], y[keep]) + tuple(e[keep] for e in extras)

print(f"Alias maps defined for {len(ALIAS_MAPS)} datasets")
for ds, am in ALIAS_MAPS.items():
    print(f"  {ds}: {len(am)} alias entries / {N_SEM} canonical = "
          f"{len(am)/N_SEM*100:.0f}% (alias entries; actual coverage verified at runtime)")
print()
print("NOTE: actual coverage verified at runtime by build_semantic_vector().")
print("Run Cell 2b (diagnostic) after Cell 6 to confirm per-dataset coverage.")
print("Short aliases (<5 chars) only match via exact alias lookup, NOT substring.")

# ── ZERO-FILL SEMANTICS NOTE (paper Section 3.4.1) ───────────────────────────
# Absent features are zero-filled: conflates "true value is zero" with
# "feature not observed". This is a known limitation, disclosed in the paper.
# Mitigation: combine()._balance() enforces 1:1 class balance within each
# dataset block, removing the shortcut-learning risk. Cell 7b verifies this
# empirically. Post-scaling zero-fill displacement is bounded by clip(±10).
# CICIDS-2017 scope note: CICIDS-2017 is a general enterprise IDS benchmark
# (Sharafaldin et al., ICISSP 2018), not an IoT-specific botnet dataset.
# It is retained for canonical vocabulary coverage (91%) and cross-extractor
# domain shift testing. See paper Section 3.3.1 and Section 6 (Limitations).
ZERO_FILL_NOTE = (
    "Absent features zero-filled — conflation with true-zero is a known "
    "limitation. See Cell 7b shortcut test and paper Section 3.4.1."
)
UNSW_SCOPE_NOTE = (
    "UNSW-NB15 is a general enterprise/network testbed (Moustafa & Slay 2015), "
    "NOT IoT-specific. Retained as a THIRD extractor family (Argus/Bro) for "
    "cross-extractor generalisation — same scope caveat as CICIDS-2017."
)
CICIDS2017_SCOPE_NOTE = (
    "CICIDS-2017 is a general enterprise IDS benchmark, NOT IoT-specific. "
    "Retained as vocabulary anchor and cross-extractor shift test only."
)
print(f"Zero-fill policy: {ZERO_FILL_NOTE}")
print(f"CICIDS-2017 scope: {CICIDS2017_SCOPE_NOTE}")

Canonical vocabulary: 46 features
Alias maps defined for 4 datasets
  CICIDS-2017: 42 alias entries / 46 canonical = 91% (alias entries; actual coverage verified at runtime)
  CIC-BCCC-NRC-TabularIoTAttacks-2024: 42 alias entries / 46 canonical = 91% (alias entries; actual coverage verified at runtime)
  CICIoMT-2024: 12 alias entries / 46 canonical = 26% (alias entries; actual coverage verified at runtime)
  UNSW-NB15: 11 alias entries / 46 canonical = 24% (alias entries; actual coverage verified at runtime)

NOTE: actual coverage verified at runtime by build_semantic_vector().
Run Cell 2b (diagnostic) after Cell 6 to confirm per-dataset coverage.
Short aliases (<5 chars) only match via exact alias lookup, NOT substring.
Zero-fill policy: Absent features zero-filled — conflation with true-zero is a known limitation. See Cell 7b shortcut test and paper Section 3.4.1.
CICIDS-2017 scope: CICIDS-2017 is a general enterprise IDS benchmark, NOT IoT-specific. Retained as vocabulary anchor an

## Configuration


In [3]:
class Config:
    # ── Kaggle dataset paths ──────────────────────────────────────────────
    # PRIMARY (flow-level, all CICFlowMeter-compatible)
    CICIDS_PATH  = "/kaggle/input/datasets/bertvankeulen/cicids-2017"
    BCCC_PATH    = "/kaggle/input/datasets/kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024"
    IOMT_PATH    = "/kaggle/input/datasets/sinfdosh1990/ciciomt2024"
    UNSW_PATH    = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"

    # Auto-detect
    USE_CICIDS  = os.path.isdir(CICIDS_PATH) if CICIDS_PATH else False
    USE_BCCC    = os.path.isdir(BCCC_PATH)   if BCCC_PATH   else False
    USE_IOMT    = os.path.isdir(IOMT_PATH)   if IOMT_PATH   else False
    USE_UNSW    = os.path.isdir(UNSW_PATH)   if UNSW_PATH   else False

    # ── Row caps per dataset ──────────────────────────────────────────────
    CICIDS_MAX   = 500_000
    BCCC_MAX     = 3_000_000   # BCCC aggregates 9 IoT datasets, 1M+ records
    IOMT_MAX     = 3_000_000
    UNSW_MAX     = 500_000     # UNSW-NB15 train+test ~257k rows (cap not hit)

    # ── Sequence parameters ───────────────────────────────────────────────
    # REVISION 2 (R3 §3.5): odd window -> no 16/16 label ties. Training windows
    # overlap (stride 4); validation, test and LODO-target windows do NOT overlap
    # (stride = window), so every evaluation window is an independent bag (R3 §3.3).
    WINDOW_SIZE = 33
    STRIDE      = 4
    EVAL_STRIDE = 33
    # REVISION 2 (R3 §3.4): rows are shuffled with this seed at load time, so a
    # window is a random bag of records from one dataset, NOT a time sequence.
    DATA_SEED   = 42
    PART        = "C1"
    TIME_BUDGET_H = 10.5   # optional experiments are skipped past this elapsed time
    MAX_TRAIN_SEQ = 800_000
    MAX_TEST_SEQ  = 200_000
    MAX_VAL_SEQ   = 200_000

    # ── Class balance ─────────────────────────────────────────────────────
    TARGET_ATK_BEN_RATIO = 1.0   # strict 1:1 record balance; odd W -> ~50% attack windows

    # ── Training ──────────────────────────────────────────────────────────
    EPOCHS        = 30
    WARMUP        = 3
    EARLY_STOP    = 7
    BATCH_SIZE    = 512
    LR            = 5e-4
    WD            = 1e-4
    FOCAL_GAMMA   = 2.5
    LABEL_SMOOTH  = 0.01
    AUX_WT        = 0.05
    DEEP_SUP_WT   = 0.3     # per-expert CE weight (deep supervision; prevents gate collapse)
    CORAL_WT      = 0.5     # source-domain CORAL alignment weight (0 disables; ablatable)
    TTBN_ADAPT    = True    # transductive test-time BatchNorm adaptation at LODO eval

    # ── Architecture ──────────────────────────────────────────────────────
    EMBED_DIM     = 32
    CONV_CH       = [64, 128, 128]
    GRU_HIDDEN    = 128
    GRU_LAYERS    = 2
    ATTN_HEADS    = 8
    DROPOUT       = 0.20
    CBGAF_DIM     = 128

    # ── Evaluation ────────────────────────────────────────────────────────
    EVAL_SEEDS     = [42, 123, 456, 789, 2024]
    FAST_SEEDS     = [42, 123, 456]
    # REVISION (R1-M1): baselines use the SAME 5 seeds as TCH-Net so every
    # in-distribution paired test is matched seed-for-seed (n = 5 pairs).
    BASE_SEEDS     = [42, 123, 456, 789, 2024]
    LODO_SEEDS     = [42, 123, 456, 789, 2024]   # REVISION 2 (R4-4): 5 seeds per fold
    ABL_EPOCHS     = 30
    BASE_EPOCHS    = 30
    LODO_EPOCHS    = 30
    WARMUP_FAST    = 5

    N_DEV_CATS = 6   # 0=sensor, 1=camera, 2=appliance, 3=IIoT, 4=server, 5=unknown

    # ── Realistic deployment prior ─────────────────────────────────
    # Attack prevalence in real IoT traffic is typically 1-10%.
    # We evaluate at 5% to simulate realistic deployment conditions.
    REALISTIC_ATK_PRIOR = 0.05   # P(attack) in realistic traffic
    OPS_PRIORS = [0.01, 0.05, 0.10]   # REVISION 2 (R3 §3.6): class-conditional reweighting

    # ── HP sensitivity ranges ──────────────────
    HP_SENS_CONFIGS = [
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.20},   # default
        {'lr': 1e-3, 'wd': 1e-4, 'dropout': 0.20},   # 2x LR
        {'lr': 2e-4, 'wd': 1e-4, 'dropout': 0.20},   # 0.4x LR
        {'lr': 5e-4, 'wd': 1e-3, 'dropout': 0.20},   # 10x WD
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.30},   # higher dropout
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.10},   # lower dropout
    ]
    HP_SENS_SEEDS = [42, 123]   # 2 seeds × 6 configs = 12 runs
    N_CLASSES  = 2   # benign vs attack (binary IDS)
    N_DS_SRC   = 4   # CICIDS-2017=0, BCCC=1, CICIoMT-2024=2, UNSW-NB15=3

    # Device category map: dataset → {device_id: category_id}
    # device_id is an arbitrary per-loader integer; category is one of N_DEV_CATS
    DEVICE_CAT_MAP = {
        "CICIDS-2017":                          {0: 4},   # 0→server/workstation
        "CIC-BCCC-NRC-TabularIoTAttacks-2024":  {1: 0},   # 1→IoT sensor/device (mixed: cameras, lights, sensors)
        "CICIoMT-2024":                         {2: 3},   # 2→IIoT/medical device
        "UNSW-NB15":                            {3: 4},   # 3→server/workstation (enterprise, non-IoT)
    }

    # ── Reviewer-requested fixes ──────────────────────────────────────────────
    # chronological split is the DEFAULT.
    SPLIT_MODE      = 'positional'   # per-dataset 70/10/20 by (shuffled) row position

    # REVISION (R1-m1): explicit validation subset for early stopping and
    # checkpoint selection. Carved chronologically per dataset BEFORE windowing:
    # train | val | test = 70 | 10 | 20 % of rows in row order. The test rows are
    # identical to the previous two-way split and are never used for selection.
    VAL_FRAC        = 0.10
    # LODO: validation is carved the same way from the SOURCE datasets only —
    # the held-out target domain is never used for early stopping.
    LODO_VAL_FRAC   = 0.10

    # feature-availability mask
    USE_MISS_MASK   = True

    # common seed set for all paired tests (REVISION R1-M1: 5 matched seeds)
    COMMON_SEEDS    = [42, 123, 456, 789, 2024]
    STAT_ALPHA      = 0.05   # family-wise level for Bonferroni-corrected paired tests

    # IDS operational metrics
    FIXED_FPR_LEVELS   = [0.001, 0.01, 0.05]
    FIXED_TPR_LEVELS   = [0.95, 0.99]  # also computes FPR@TPR99 for every model

    OUT = "tch_net_v2_results"

    # ── LODO baseline names (5 DL models run in LODO alongside TCH-Net) ────
    # Transformer-IDS, BiGRU-IDS, MLP-IDS, CNN-LSTM, 1D-CNN-IDS
    LODO_BASELINE_NAMES = [
        "Transformer-IDS",
        "BiGRU-IDS",
        "MLP-IDS",
        "CNN-LSTM",
        "1D-CNN-IDS",
    ]

os.makedirs(Config.OUT, exist_ok=True)
PAL = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
       '#00BCD4','#795548','#E91E63','#607D8B','#CDDC39']
def SAVE(n, fig=None):
    """Save current figure and display inline in Jupyter."""
    _f = fig if fig is not None else plt.gcf()
    _f.savefig(os.path.join(Config.OUT, n), bbox_inches='tight', dpi=150)

print("Dataset availability:")
for n, f in [("CICIDS-2017", Config.USE_CICIDS),
             ("CIC-BCCC-NRC-TabularIoTAttacks-2024", Config.USE_BCCC),
             ("CICIoMT-2024", Config.USE_IOMT),
             ("UNSW-NB15", Config.USE_UNSW)]:
    print(f"  {n:<38}: {'FOUND' if f else 'NOT FOUND'} [PRIMARY]")

Dataset availability:
  CICIDS-2017                           : FOUND [PRIMARY]
  CIC-BCCC-NRC-TabularIoTAttacks-2024   : FOUND [PRIMARY]
  CICIoMT-2024                          : FOUND [PRIMARY]
  UNSW-NB15                             : FOUND [PRIMARY]


## Dataset Loaders

Each loader:
1. Finds and reads CSV/Parquet files
2. Identifies label column and converts to binary (0=benign, 1=attack)
3. Calls `build_semantic_vector()` to map to the 46-feature canonical space
4. Reports coverage: how many of the 46 features were matched
5. Returns `(X_semantic, y, device_ids)`

**No fabricated mappings.** If a dataset lacks `iat_mean`, that column stays zero.


In [4]:
ATTACK_TYPE_STORE = {}

def _subsample(X, y, max_n, extras=None, seed=42):
    """
    Subsample to at most max_n rows, preserving the original class ratio
    but enforcing a minimum of 5000 samples per class when possible.
    The actual class re-balancing to 1:1 happens later in combine()._balance().
    This function only caps extremely large datasets to avoid OOM.
    """
    if max_n is None or len(X) <= max_n:
        return (X, y) + (tuple(extras) if extras else ())
    rng = np.random.RandomState(seed)
    ben = np.where(y==0)[0]; atk = np.where(y==1)[0]
    r = len(ben) / len(y)
    nb = min(len(ben), max(5_000, int(max_n * r)))
    na = min(len(atk), max(5_000, max_n - nb))
    if nb + na > max_n:
        scale = max_n / (nb + na)
        nb = max(5_000, int(nb * scale))
        na = max(5_000, min(max_n - nb, int(na * scale)))
    nb = min(nb, len(ben)); na = min(na, len(atk))
    keep = np.concatenate([rng.choice(ben, nb, replace=False),
                           rng.choice(atk, na, replace=False)])
    rng.shuffle(keep)
    result = [X[keep], y[keep]]
    if extras:
        for a in extras: result.append(a[keep])
    print(f"  Subsampled {len(X):,} -> {len(keep):,} "
          f"(ben={nb:,} atk={na:,} atk%={na/(nb+na)*100:.1f}%)")
    return tuple(result)

def augment_minority(X, y, target=5000, noise_std=0.01, seed=42):
    rng = np.random.RandomState(seed)
    ben_idx = np.where(y == 0)[0]
    atk_idx = np.where(y == 1)[0]
    minority_idx = ben_idx if len(ben_idx) < len(atk_idx) else atk_idx
    minority_label = 0 if len(ben_idx) < len(atk_idx) else 1
    n_minority = len(minority_idx)
    if n_minority >= target:
        return X, y
    n_needed = target - n_minority
    print(f"    Augmenting minority ({n_minority} -> {target}, "
          f"generating {n_needed} synthetic rows via Gaussian noise sigma={noise_std})")
    chosen = rng.choice(minority_idx, n_needed, replace=True)
    X_aug = X[chosen] + rng.normal(0, noise_std,
                                    size=(n_needed, X.shape[1])).astype(np.float32)
    X_aug = np.clip(X_aug, X.min(axis=0), X.max(axis=0))
    y_aug = np.full(n_needed, minority_label, dtype=np.int32)
    X_new = np.vstack([X, X_aug])
    y_new = np.hstack([y, y_aug])
    print(f"    After augmentation: ben={(y_new==0).sum():,} atk={(y_new==1).sum():,}")
    return X_new, y_new

def _is_benign_mask(lbl_series):
    """
    Robust benign/attack split for an object-dtype label column.
    Matches an exact small set of known benign tokens AND any label that
    merely CONTAINS "benign" (case-insensitive) — covers "Benign",
    "BENIGN", "BenignTraffic", "Benign Traffic", "Benign_Traffic", etc.
    without having to enumerate every uploader's exact string.
    """
    s = lbl_series.astype(str).str.strip().str.lower()
    exact = s.isin(['benign', 'benigntraffic', 'normal', '0'])
    contains = s.str.contains('benign', regex=False)
    return exact | contains

# ── CICIDS-2017 ──────────────────────────────────────────────────────────
# Verified against: cicids-2017-friday.csv, cicids-2017-friday-plus.csv
# (89 / 105 columns — "-plus" adds extra Flow ID / "Local_*" metadata columns
# that are not part of the canonical feature set and are safely ignored).
# Label column: "Label" (string, e.g. "BENIGN"). Uses ALIAS_FLOWMETER.
def load_cicids2017(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIDS-2017\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True) +
                   glob.glob(os.path.join(data_dir,"**","*.parquet"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description'])]
    # REVISION 2: the Kaggle mirror ships every day twice (day.csv and day_plus.csv,
    # the latter with extra metadata columns). Loading both duplicated every flow.
    files = [f for f in files if '_plus' not in os.path.basename(f).lower()]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    for fp in files:
        print(f"  Loading {os.path.basename(fp)}...")
        try:
            df = pd.read_parquet(fp) if fp.endswith('.parquet') else pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None: print(f"    No label column"); continue
            y = (~_is_benign_mask(df[lc])).values.astype(np.int32)
            atk = np.where(y==0, 'BENIGN', df[lc].astype(str).str.strip().values)
            X_sem, matched = build_semantic_vector(df, "CICIDS-2017")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_sem = apply_units_and_derivations(X_sem, "CICIDS-2017")
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            print(f"    {len(y):,} rows | coverage={len(matched)}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
        except Exception as e: print(f"    Skip: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 0 → server/workstation category (Config.DEVICE_CAT_MAP["CICIDS-2017"])
    dev = np.zeros(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CICIDS-2017'] = atk
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
    X, y, dev = dedup_canonical(X, y, 'CICIDS-2017', [dev])
    perm = np.random.RandomState(Config.DATA_SEED).permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev], seed=Config.DATA_SEED)
    return X, y, dev

# ── CIC-BCCC-NRC-TabularIoTAttacks-2024 ──────────────────────────────────
# Replaces the broken CIC-IoT-DIAD-2024 (Mostafa Ali Ogab) — that dataset's
# Label column was "NeedManualLabel" on EVERY row, including attack-named
# files. Confirmed unusable; see Cell 2 dataset-reset note.
#
# BCCC research center (York University). CICFlowMeter-extracted, 85
# columns, aggregates 9 IoT datasets into one unified table. 1M+ records.
# Verified against: cic-bccc-nrc-tabulariotAttack.csv
#   Label column: "Label" — int, 0 in the (benign-only) sample.
#   "Attack Name" column: string, e.g. "Benign Traffic" in the sample.
# Label strategy: prefer the explicit "Attack Name" string (unambiguous
# human-readable family) when present; fall back to numeric Label
# (0=benign / nonzero=attack — works whether Label is strictly binary or a
# multi-class integer encoding, since != 0 is correct in both cases).
# Kaggle: kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024
def load_cic_bccc_nrc(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CIC-BCCC-NRC-TabularIoTAttacks-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    _agree_n = 0; _agree_match = 0   # label cross-validation tally (see below)
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True

            attack_col = next((c for c in df.columns if c.lower().strip() == 'attack name'), None)
            label_col  = next((c for c in df.columns if c.lower().strip() == 'label'), None)

            # ── Label cross-validation ──────────────────────────────────────
            # This dataset carries TWO independent label signals per row:
            # a human-readable family string ("Attack Name") and a numeric
            # code ("Label"). Rather than trusting one and discarding the
            # other, derive y from BOTH whenever both are present and report
            # their agreement rate. Attack Name is used as the actual ground
            # truth (explicit, unambiguous string) — Label is a corroborating
            # signal, not a silent fallback. Any disagreement is surfaced
            # immediately instead of being silently absorbed into the labels.
            if attack_col is not None and label_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y_from_attack = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                lbl = df[label_col]
                if lbl.dtype == object:
                    y_from_label = (~_is_benign_mask(lbl)).values.astype(np.int32)
                else:
                    y_from_label = (lbl.values != 0).astype(np.int32)
                _agree = (y_from_attack == y_from_label)
                _agree_n += len(_agree); _agree_match += int(_agree.sum())
                if not _agree.all():
                    print(f"    NOTE: {int((~_agree).sum())}/{len(_agree)} rows in "
                          f"{os.path.basename(fp)} disagree between 'Attack Name' "
                          f"and numeric 'Label' — using 'Attack Name'")
                y = y_from_attack
                atk = atk_str.values
            elif attack_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                atk = atk_str.values
            elif label_col is not None:
                lbl = df[label_col]
                if lbl.dtype == object:
                    y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                    atk = lbl.astype(str).values
                else:
                    y = (lbl.values != 0).astype(np.int32)
                    atk = np.where(y==0, 'Benign Traffic', lbl.astype(str).values)
            else:
                print(f"  No 'Attack Name' or 'Label' column in {os.path.basename(fp)} — skip")
                continue

            X_sem, matched = build_semantic_vector(df, "CIC-BCCC-NRC-TabularIoTAttacks-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_sem = apply_units_and_derivations(X_sem, "CIC-BCCC-NRC-TabularIoTAttacks-2024")
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    if _agree_n > 0:
        _agree_pct = _agree_match / _agree_n * 100
        print(f"\n  Label cross-validation ('Attack Name' vs numeric 'Label'): "
              f"{_agree_match:,}/{_agree_n:,} rows agree ({_agree_pct:.2f}%)")
        if _agree_pct < 99.0:
            print(f"  WARNING: >1% disagreement between the two label columns — "
                  f"inspect before trusting either derivation in the paper.")
    # device_id 1 → generic IoT device category (BCCC mixes cameras/lights/sensors)
    dev = np.ones(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CIC-BCCC-NRC-TabularIoTAttacks-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    X, y, dev = dedup_canonical(X, y, 'CIC-BCCC-NRC-TabularIoTAttacks-2024', [dev])
    rng_bccc = np.random.RandomState(Config.DATA_SEED)
    perm = rng_bccc.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev], seed=Config.DATA_SEED)
    return X, y, dev

# ── CICIoMT-2024 (Train-Test, D. Kilichev) ───────────────────────────────
# Replaces the broken CICIoMT-2024 (Lian Mollick) variant — that upload had
# 45 columns and NO label column at all (same reduced schema as CIC-IoT-2023).
# Confirmed unusable; see Cell 2 dataset-reset note.
#
# Lashkari et al. (2024). Custom CIC statistical extractor (46-col schema
# incl. Label — identical feature schema to the broken variant, but THIS
# upload's Label column is genuinely populated).
# Verified against: ciciomt-train-set.csv ("DoS-TCP"), ciciomt-test-set.csv
# ("Malformed_Data"). Neither 10-row sample contains a benign row, so the
# exact benign string token is unconfirmed — _is_benign_mask() matches any
# label containing "benign" case-insensitively to stay robust to whichever
# exact spelling ("Benign" / "BenignTraffic" / "Benign Traffic") this
# upload uses.
# Label column: "Label". 17/46 features alias-resolve; 16/46 (35%) carry
# non-trivial signal on the full live dataset (Drate/bwd_pkt_rate resolves
# correctly but is ~0 across virtually every real row — matches the source
# paper's own published stats, not a mapping defect). See Cell 2 for the
# full Srate/Drate/Duration reasoning.
def load_ciciomt(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIoMT-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None:
                # Some file splits carry no label — skip silently
                continue
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                atk = lbl.astype(str).values
            else:
                y = (lbl.values != 0).astype(np.int32)
                atk = np.where(y==0, 'Benign', lbl.astype(str).values)
            X_sem, matched = build_semantic_vector(df, "CICIoMT-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_sem = apply_units_and_derivations(X_sem, "CICIoMT-2024")
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 2 → IIoT/medical device category
    dev = np.full(len(y), 2, dtype=np.int32)
    ATTACK_TYPE_STORE['CICIoMT-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    X, y, dev = dedup_canonical(X, y, 'CICIoMT-2024', [dev])
    rng_iomt = np.random.RandomState(Config.DATA_SEED)
    perm = rng_iomt.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev], seed=Config.DATA_SEED)
    return X, y, dev

# \u2500\u2500 UNSW-NB15 (Moustafa & Slay 2015; Argus + Bro/Zeek) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# THIRD extractor family \u2014 the heterogeneity upgrade. Enterprise/general
# network testbed (NOT IoT-specific \u2014 same scope caveat as CICIDS-2017).
# Reads ONLY the clean labelled split (files exposing BOTH a `label` and a
# `dur` header column): UNSW_NB15_training-set.csv / _testing-set.csv.
# This automatically SKIPS:
#   * the raw headerless captures UNSW-NB15_1..4.csv (no header -> first data
#     row misread as header -> lacks `label`/`dur` -> rejected), and
#   * NUSW-NB15_features.csv / *LIST_EVENTS* metadata files.
# Label: numeric `label` (0=normal, 1=attack). `attack_cat` gives the family.
# Uses ALIAS_UNSW. device_id 3 -> enterprise/server category (DEVICE_CAT_MAP).
# Kaggle: mrwellsdavid/unsw-nb15
def load_unsw_nb15(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING UNSW-NB15\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description','list_events',
                        'list-events','ground_truth','_gt'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            cols_lower = {c.lower() for c in df.columns}
            # Guard: accept ONLY the clean labelled split. Raw headerless
            # captures and the features-metadata file lack this exact pair.
            if 'label' not in cols_lower or 'dur' not in cols_lower:
                print(f"  Skip {os.path.basename(fp)}: not the clean labelled "
                      f"schema (needs 'label'+'dur' header) \u2014 raw/meta file")
                continue
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next(c for c in df.columns if c.lower().strip() == 'label')
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
            else:
                y = (lbl.values != 0).astype(np.int32)   # 0=normal, 1=attack
            ac = next((c for c in df.columns if c.lower().strip() == 'attack_cat'), None)
            if ac is not None:
                atk = np.where(y==0, 'Normal', df[ac].astype(str).str.strip().values)
            else:
                atk = np.where(y==0, 'Normal', 'Attack')
            X_sem, matched = build_semantic_vector(df, "UNSW-NB15")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_sem = apply_units_and_derivations(X_sem, "UNSW-NB15")
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
        except Exception as e:
            print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 3 -> enterprise/server category (non-IoT, like CICIDS-2017)
    dev = np.full(len(y), 3, dtype=np.int32)
    ATTACK_TYPE_STORE['UNSW-NB15'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5 or atk_pct > 95:
        print(f"  WARNING: extreme attack% ({atk_pct:.1f}%) \u2014 check label parsing")
    X, y, dev = dedup_canonical(X, y, 'UNSW-NB15', [dev])
    rng_unsw = np.random.RandomState(Config.DATA_SEED)
    perm = rng_unsw.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev], seed=Config.DATA_SEED)
    return X, y, dev

print("All 4 dataset loaders defined")

All 4 dataset loaders defined


## Multi-Dataset Loader (Leakage-Free)

**Pipeline:**
1. Balance each dataset independently (strict 1:1)
2. Skip datasets with 0% feature coverage
3. Chronological 80/20 split per dataset (default) — prevents adjacent-window leakage
4. Fit single RobustScaler on combined train split
5. Transform both splits; clip to [-10, 10]

**Context label:** Each sample carries `[dataset_source_id, device_category_id]`, used only as the CORAL domain label during training and for LODO fold bookkeeping — it is not consumed as a network input.


In [5]:
class MultiDatasetLoader:
    # Warn if combined post-balance count < 2*MIN_FLOOR (signals scarce dataset).
    # Any dataset whose minority class is small will be flagged but still included.
    MIN_FLOOR = 5_000  # advisory threshold only

    def __init__(self):
        self.datasets = {}
        self.scaler = None
        # NOTE: X_train_scaled / X_test_scaled (stored after scaling)
        # Named *_raw to match usage in HP sensitivity cells.
        self.X_train_raw = self.X_test_raw = self.X_val_raw = None

    def add(self, name, X, y, dev, ds_src_id):
        if X is None: print(f"  Skip {name}: no data"); return
        cat_map = Config.DEVICE_CAT_MAP.get(name, {})
        dev_cats = np.array([cat_map.get(int(d), 5) for d in dev], dtype=np.int32)
        ds_ids   = np.full(len(y), ds_src_id, dtype=np.int32)
        ctx      = np.stack([ds_ids, dev_cats], axis=1)
        n_matched = int((X != 0).any(axis=0).sum())
        self.datasets[name] = {'X': X, 'y': y, 'dev': dev, 'ctx': ctx,
                               'ds_src': ds_src_id, 'n': X.shape[0]}
        print(f"  Added {name}: {X.shape[0]:,}x{X.shape[1]} "
              f"(non-zero dims: {n_matched}/{N_SEM} = {n_matched/N_SEM*100:.0f}%)")

    def _balance(self, X, y, ctx, ratio, rng, name=""):
        """
        Strict class balance with seeded RNG for reproducibility.

        ratio=1.0 → 50% attack rows → p(window=attack) ≈ 43% with majority-vote(w=32).

        MIN_FLOOR: advisory only — warns when a dataset contributes fewer than
        2*MIN_FLOOR total samples after balancing (poor signal, still included).
        The actual cap is ALWAYS strict: majority capped to ratio * minority.
        We never inflate the majority beyond ratio*minority just because minority
        is small — that would produce up to 5:1 imbalance and near-100% attack
        windows after windowing (empirically verified on prior, since-removed
        low-sample datasets in this pipeline's history).
        FocalLoss alpha weighting handles residual class imbalance at batch level.
        """
        ben = np.where(y == 0)[0]
        atk = np.where(y == 1)[0]
        if len(ben) == 0 or len(atk) == 0:
            print(f"  WARNING {name}: single-class data "
                  f"(ben={len(ben)}, atk={len(atk)}) — skipping balance")
            return X, y, ctx

        # Strict 1:1 balance always — even if minority class is tiny.
        # Reason: majority-vote windowing needs ~50% row balance to produce
        # mixed-label windows. A 20:1 imbalance produces ~100% attack windows,
        # giving the model zero benign examples from that dataset.
        # A dataset with very few minority-class rows is accepted as-is rather
        # than upsampled past this floor — FocalLoss alpha handles the residual.
        # FocalLoss alpha and the global _cap handle residual imbalance.
        MIN_SAMPLES = 5000
        if len(atk) > ratio * len(ben):
            if len(ben) >= MIN_SAMPLES:
                n_atk = int(ratio * len(ben))
                atk = rng.choice(atk, n_atk, replace=False)
            else:
                n_atk = min(len(atk), MIN_SAMPLES)
                atk = rng.choice(atk, n_atk, replace=False)
        elif len(ben) > ratio * len(atk):
            if len(atk) >= MIN_SAMPLES:
                n_ben = int(ratio * len(atk))
                ben = rng.choice(ben, n_ben, replace=False)
            else:
                n_ben = min(len(ben), MIN_SAMPLES)
                ben = rng.choice(ben, n_ben, replace=False)

        n_total = len(ben) + len(atk)
        if n_total < 2 * self.MIN_FLOOR:
            print(f"  ADVISORY {name}: only {n_total:,} samples after balancing "
                  f"(minority class was very small: {min(len(ben),len(atk)):,}). "
                  f"FocalLoss alpha weighting compensates during training.")

        keep = np.concatenate([ben, atk])
        # CRITICAL FIX: DO NOT shuffle here.
        # combine() performs a chronological split by sorting positional indices
        # within each dataset block. Shuffling here destroys temporal order
        # within the block, making ds_idx_sorted meaningless as a time-order
        # proxy. Instead, sort keep so rows are in their original temporal order.
        # _cap() shuffles windows after sequencing — randomness is preserved there.
        keep = np.sort(keep)
        return X[keep], y[keep], ctx[keep]

    def combine(self, ratio=None, test_size=0.2, seed=None):
        if ratio is None: ratio = Config.TARGET_ATK_BEN_RATIO
        if seed is None: seed = Config.DATA_SEED
        rng = np.random.RandomState(seed)
        print("\n" + "="*70 + "\nCOMBINE & PREPROCESS\n" + "="*70)
        Xa, ya, ca, da = [], [], [], []

        for di, (name, d) in enumerate(self.datasets.items()):
            X_r = d['X'].copy(); y_r = d['y'].copy(); c_r = d['ctx'].copy()
            np.nan_to_num(X_r, copy=False, nan=0., posinf=1e6, neginf=-1e6)
            # REVISION 2: re-shuffle rows with the combine seed, so different data
            # seeds give different balance draws AND different partitions (R4-4).
            if seed != Config.DATA_SEED:
                _p = np.random.RandomState(seed).permutation(len(y_r))
                X_r, y_r, c_r = X_r[_p], y_r[_p], c_r[_p]

            n_nonzero = int((X_r != 0).any(axis=0).sum())
            if n_nonzero == 0:
                print(f"  SKIP {name}: 0% feature coverage — all features zero.")
                continue
            if n_nonzero < 5:
                print(f"  WARNING {name}: very low coverage ({n_nonzero}/46). Results unreliable.")

            # ── FIX: derive pkt_count_total BEFORE scaling ───────────────────
            # Cannot do fwd_scaled + bwd_scaled ≠ scale(fwd+bwd). Must use raw counts.
            _fwd = SEM_IDX["pkt_count_fwd"]
            _bwd = SEM_IDX["pkt_count_bwd"]
            _tot = SEM_IDX["pkt_count_total"]
            zero_tot = (X_r[:, _tot] == 0)
            X_r[zero_tot, _tot] = X_r[zero_tot, _fwd] + X_r[zero_tot, _bwd]

            X_r, y_r, c_r = self._balance(X_r, y_r, c_r, ratio, rng, name)
            ben_r = int((y_r==0).sum()); atk_r = int((y_r==1).sum())
            est_seq = max(0, (len(y_r) - Config.WINDOW_SIZE)) // Config.STRIDE + 1
            print(f"  {name}: {len(y_r):,} rows "
                  f"(ben={ben_r:,} atk={atk_r:,} "
                  f"ratio={atk_r/max(ben_r,1):.1f}:1) "
                  f"est_sequences~{est_seq:,}")
            Xa.append(X_r); ya.append(y_r); ca.append(c_r)
            da.append(np.full(len(y_r), di, dtype=np.int32))

        if not Xa:
            raise RuntimeError("No datasets survived combine() — check feature coverage")

        X_c   = np.vstack(Xa).astype(np.float32)
        y_c   = np.hstack(ya).astype(np.int32)
        ctx_c = np.vstack(ca).astype(np.int32)
        ds_c  = np.hstack(da).astype(np.int32)

        total_ben = int((y_c == 0).sum()); total_atk = int((y_c == 1).sum())
        print(f"\n  Pre-split combined: {len(y_c):,} rows "
              f"| ben={total_ben:,} ({total_ben/len(y_c)*100:.1f}%) "
              f"| atk={total_atk:,} ({total_atk/len(y_c)*100:.1f}%)")

        # ── CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT (default) ──────────
        # REVISION (R1-m1). Split BEFORE window generation, per dataset, by
        # original row order:
        #   first (1 − test_size − VAL_FRAC) of rows → train
        #   next   VAL_FRAC                         → validation (early stopping
        #                                             and checkpoint selection ONLY)
        #   last   test_size                        → test (same rows as the
        #                                             previous two-way split)
        # Windows are generated separately inside each partition and never across
        # datasets, so no sliding window can contain rows from two partitions:
        # overlapping-window leakage between train, validation and test is
        # impossible by construction. Previously there was no validation subset
        # and early stopping monitored the test set.
        _vf = getattr(Config, 'VAL_FRAC', 0.10)
        if Config.SPLIT_MODE in ('positional', 'chronological'):
            i_tr, i_va, i_te = [], [], []
            for ds_id in np.unique(ds_c):
                # _balance() keeps rows in original (temporal) order and datasets
                # are concatenated block-wise, so sorted positions = time order.
                ds_idx_sorted = np.sort(np.where(ds_c == ds_id)[0])
                n_trva = int(len(ds_idx_sorted) * (1 - test_size))
                n_va   = int(len(ds_idx_sorted) * _vf)
                i_tr.append(ds_idx_sorted[:n_trva - n_va])
                i_va.append(ds_idx_sorted[n_trva - n_va:n_trva])
                i_te.append(ds_idx_sorted[n_trva:])
            i_tr, i_va, i_te = np.concatenate(i_tr), np.concatenate(i_va), np.concatenate(i_te)
            print(f"  Split mode: POSITIONAL on seeded-shuffled rows (per-dataset train/val/test = "
                  f"{(1-test_size-_vf)*100:.0f}/{_vf*100:.0f}/{test_size*100:.0f}% by row order)")
        else:
            # Random split retained for ablation comparisons only
            _idx_all = np.arange(len(y_c))
            _idx_trva, i_te = train_test_split(_idx_all, test_size=test_size,
                                               stratify=y_c, random_state=seed)
            i_tr, i_va = train_test_split(_idx_trva, test_size=_vf / (1 - test_size),
                                          stratify=y_c[_idx_trva], random_state=seed)
            print(f"  Split mode: RANDOM stratified (ablation only — not default)")
        X_tr, y_tr, c_tr, d_tr = X_c[i_tr], y_c[i_tr], ctx_c[i_tr], ds_c[i_tr]
        X_va, y_va, c_va, d_va = X_c[i_va], y_c[i_va], ctx_c[i_va], ds_c[i_va]
        X_te, y_te, c_te, d_te = X_c[i_te], y_c[i_te], ctx_c[i_te], ds_c[i_te]

        # Scale AFTER pkt_count_total derivation (now done above, pre-split).
        # The scaler is fit on TRAIN rows only; validation and test are transformed.
        self.scaler = RobustScaler(quantile_range=(5, 95))
        X_tr = np.clip(self.scaler.fit_transform(X_tr), -10, 10).astype(np.float32)
        X_va = np.clip(self.scaler.transform(X_va),     -10, 10).astype(np.float32)
        X_te = np.clip(self.scaler.transform(X_te),     -10, 10).astype(np.float32)

        # Validate post-split balance
        for tag_, y_ in [('Train', y_tr), ('Val', y_va), ('Test', y_te)]:
            b_ = int((y_==0).sum()); a_ = int((y_==1).sum())
            print(f"  {tag_+':':<6} {len(y_):,} | ben={b_:,} ({b_/len(y_)*100:.1f}%) "
                  f"| atk={a_:,} ({a_/len(y_)*100:.1f}%)")
        # Per-dataset validation composition (disclosed in the paper's split description)
        _names_ = list(self.datasets.keys())
        for ds_id in np.unique(d_va):
            _m_ = (d_va == ds_id)
            _nm_ = _names_[ds_id] if ds_id < len(_names_) else f'DS{ds_id}'
            print(f"    val rows {_nm_:<38}: {int(_m_.sum()):,} "
                  f"(attack {float(y_va[_m_].mean())*100:.1f}%)")

        # Store scaled data for HP sensitivity reuse
        self.X_train_raw = X_tr; self.y_train_raw = y_tr
        self.ctx_train_raw = c_tr; self.ds_train_raw = d_tr
        self.X_val_raw   = X_va; self.y_val_raw   = y_va
        self.ctx_val_raw = c_va; self.ds_val_raw = d_va
        self.X_test_raw  = X_te; self.y_test_raw  = y_te
        self.ctx_test_raw = c_te; self.ds_test_raw = d_te

        print(f"  Scaler: single RobustScaler(q5,q95) fit on train only")
        # Post-split balance check
        for tag_, y_ in [('Train', y_tr), ('Test', y_te)]:
            r_ = float((y_==1).mean())
            if r_ < 0.05 or r_ > 0.95:
                raise RuntimeError(f'{tag_} set is severely imbalanced: attack%={r_*100:.1f}%. Check _balance().')
        _rv = float((y_va==1).mean())
        if _rv < 0.05 or _rv > 0.95:
            print(f"  WARNING: validation set is imbalanced (attack%={_rv*100:.1f}%); "
                  f"early stopping falls back to validation loss if it is single-class.")
        return X_tr, y_tr, c_tr, d_tr, X_te, y_te, c_te, d_te

loader = MultiDatasetLoader()
DS_SRC = {
    "CICIDS-2017":                          0,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  1,
    "CICIoMT-2024":                         2,
    "UNSW-NB15":                             3,
}
print("MultiDatasetLoader ready")
print(f"Dataset source IDs: {DS_SRC}")

MultiDatasetLoader ready
Dataset source IDs: {'CICIDS-2017': 0, 'CIC-BCCC-NRC-TabularIoTAttacks-2024': 1, 'CICIoMT-2024': 2, 'UNSW-NB15': 3}


## Load All Datasets


In [6]:
print("="*70+"\nLOADING DATASETS\n"+"="*70)

if Config.USE_CICIDS:
    try:
        X,y,d = load_cicids2017(Config.CICIDS_PATH, Config.CICIDS_MAX)
        if X is not None: loader.add('CICIDS-2017', X, y, d, DS_SRC['CICIDS-2017'])
        del X,y,d
    except Exception as e: print(f"CICIDS-2017 failed: {e}")

if Config.USE_BCCC:
    try:
        X,y,d = load_cic_bccc_nrc(Config.BCCC_PATH, Config.BCCC_MAX)
        if X is not None: loader.add('CIC-BCCC-NRC-TabularIoTAttacks-2024', X, y, d,
                                      DS_SRC['CIC-BCCC-NRC-TabularIoTAttacks-2024'])
        del X,y,d
    except Exception as e: print(f"CIC-BCCC-NRC-TabularIoTAttacks-2024 failed: {e}")

if Config.USE_IOMT:
    try:
        X,y,d = load_ciciomt(Config.IOMT_PATH, Config.IOMT_MAX)
        if X is not None: loader.add('CICIoMT-2024', X, y, d, DS_SRC['CICIoMT-2024'])
        del X,y,d
    except Exception as e: print(f"CICIoMT-2024 failed: {e}")

if Config.USE_UNSW:
    try:
        X,y,d = load_unsw_nb15(Config.UNSW_PATH, Config.UNSW_MAX)
        if X is not None: loader.add('UNSW-NB15', X, y, d, DS_SRC['UNSW-NB15'])
        del X,y,d
    except Exception as e: print(f"UNSW-NB15 failed: {e}")

gc.collect()
DS_NAMES = list(loader.datasets.keys())
print(f"\n{len(DS_NAMES)} datasets loaded: {DS_NAMES}")
assert len(DS_NAMES) >= 1, "No datasets loaded — check Kaggle input paths in Config"

# ── Feature coverage table (for paper) ───────────────────────────────────
print(f"\n{'='*70}\nFEATURE COVERAGE TABLE (for Section III of paper)\n{'='*70}")
print(f"{'Dataset':<38} {'Matched':>8} {'Coverage':>10} {'Zero-filled features (first 5)'}")
print("-"*70)
for name, data in loader.datasets.items():
    X = data['X']
    covered = int((X != 0).any(axis=0).sum())
    zero_feats = [SEMANTIC_FEATURES[i] for i in range(N_SEM) if (X[:, i] == 0).all()]
    z_str = ', '.join(zero_feats[:5]) + ('...' if len(zero_feats)>5 else '')
    print(f"  {name:<38} {covered:>5}/{N_SEM}  {covered/N_SEM*100:>7.0f}%   {z_str}")


LOADING DATASETS

LOADING CICIDS-2017


  Loading friday.csv...


    547,557 rows | coverage=42/46 | Ben=288,544 Atk=259,013
  Loading monday.csv...


    371,621 rows | coverage=42/46 | Ben=371,621 Atk=0
  Loading thursday.csv...


    362,075 rows | coverage=42/46 | Ben=288,171 Atk=73,904
  Loading tuesday.csv...


...

  Columns (45): ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit']
    ... and 25 more
  UNSW_NB15_testing-set.csv: 175,341 rows | cov=11/46 | Ben=56,000 Atk=119,341 | acc=175,341


  UNSW_NB15_training-set.csv: 82,332 rows | cov=11/46 | Ben=37,000 Atk=45,332 | acc=257,673
  Total: 257,673 | Ben=93,000 Atk=164,673 (63.9% attack)
  De-duplication: 257,673 rows -> 132,528 unique, label-consistent vectors (48.4% duplicates, 554 conflicting vectors dropped)


  Added UNSW-NB15: 132,528x46 (non-zero dims: 19/46 = 41%)

4 datasets loaded: ['CICIDS-2017', 'CIC-BCCC-NRC-TabularIoTAttacks-2024', 'CICIoMT-2024', 'UNSW-NB15']

FEATURE COVERAGE TABLE (for Section III of paper)
Dataset                                 Matched   Coverage Zero-filled features (first 5)
----------------------------------------------------------------------
  CICIDS-2017                               46/46      100%   


  CIC-BCCC-NRC-TabularIoTAttacks-2024       46/46      100%   


  CICIoMT-2024                              12/46       26%   flow_duration, pkt_count_fwd, pkt_count_bwd, byte_count_fwd, byte_count_bwd...
  UNSW-NB15                                 19/46       41%   subflow_fwd_pkts, subflow_bwd_pkts, pkt_len_min, pkt_len_max, pkt_len_mean...


## Data report: units, derived slots, de-duplication, coverage tiers
Revision 2 (Reviewer 3 §3.2, §3.7; Reviewer 4 comment 1). Coverage is reported in four tiers:
**mapped** (schema present), **derived** (computed from mapped slots with the canonical definition),
**non-constant** (carries any variation), and **independent** (non-constant after removing columns
that are exact copies of another column).

In [7]:
RUN_T0 = time.time()
DATA_REPORT = {'unit_table': UNIT_TABLE, 'dedup': DEDUP_STATS, 'derived_slots': DERIVED_SLOTS,
               'alias_maps': {k: {s: v[0] for s, v in m.items()} for k, m in ALIAS_MAPS.items()},
               'coverage': {}}
COVERED = {}
print('=' * 70 + '\nCOVERAGE TIERS\n' + '=' * 70)
print(f"  {'dataset':<38} {'mapped':>7} {'derived':>8} {'non-const':>10} {'independent':>12}")
for _n, _d in loader.datasets.items():
    _X = _d['X']
    _mapped = sorted(ALIAS_MAPS.get(_n, {}).keys())
    _derived = DERIVED_SLOTS.get(_n, [])
    _nonconst = [SEMANTIC_FEATURES[j] for j in range(N_SEM) if np.nanstd(_X[:, j]) > 0]
    # independent: drop columns that duplicate an earlier non-constant column exactly
    _seen, _indep = [], []
    for f in _nonconst:
        col = _X[:, SEM_IDX[f]]
        if any(np.array_equal(col, _X[:, SEM_IDX[g]]) for g in _seen):
            continue
        _seen.append(f); _indep.append(f)
    COVERED[_n] = _nonconst
    DATA_REPORT['coverage'][_n] = {'mapped': _mapped, 'derived': _derived,
                                   'non_constant': _nonconst, 'independent': _indep}
    print(f"  {_n:<38} {len(_mapped):>7} {len(_derived):>8} {len(_nonconst):>10} {len(_indep):>12}")
_common_all = [f for f in SEMANTIC_FEATURES if all(f in COVERED[n] for n in COVERED)]
_common_3 = [f for f in SEMANTIC_FEATURES if sum(f in COVERED[n] for n in COVERED) >= max(len(COVERED) - 1, 1)]
DATA_REPORT['common_features_all'] = _common_all
DATA_REPORT['common_features_all_but_one'] = _common_3
print(f"\n  Features non-constant in ALL datasets ({len(_common_all)}): {_common_all}")
print(f"  ... in all but one dataset ({len(_common_3)}): {_common_3}")
print('\n  De-duplication:', json.dumps(DEDUP_STATS, indent=1))

COVERAGE TIERS
  dataset                                 mapped  derived  non-const  independent


  CICIDS-2017                                 42        4         46           44


  CIC-BCCC-NRC-TabularIoTAttacks-2024         42        4         46           44


  CICIoMT-2024                                12        0         12           12
  UNSW-NB15                                   11        8         19           17

  Features non-constant in ALL datasets (3): ['pkt_rate', 'pkt_count_total', 'byte_count_total']
  ... in all but one dataset (28): ['flow_duration', 'pkt_count_fwd', 'pkt_count_bwd', 'byte_count_fwd', 'byte_count_bwd', 'pkt_rate', 'byte_rate', 'fwd_pkt_rate', 'bwd_pkt_rate', 'fwd_byte_rate', 'bwd_byte_rate', 'pkt_count_total', 'byte_count_total', 'fwd_pkt_len_total', 'bwd_pkt_len_total', 'pkt_len_min', 'pkt_len_max', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean', 'fwd_iat_mean', 'bwd_iat_mean', 'flag_syn', 'flag_ack', 'flag_fin', 'flag_rst']

  De-duplication: {
 "CICIDS-2017": {
  "rows_in": 2099971,
  "unique_vectors": 1713330,
  "conflicting_vectors_dropped": 15,
  "rows_kept": 1713315,
  "duplicate_frac": 0.18411730447706187
 },
 "CIC-BCCC-NRC-TabularIoTAttacks-2024": {
  "rows_in

## Combine, Scale & Create Sequences


In [8]:
(X_train_raw, y_train_raw, ctx_train_raw, ds_train_raw,
 X_test_raw,  y_test_raw,  ctx_test_raw,  ds_test_raw) = loader.combine()

N_FEATURES = N_SEM
# NOTE: pkt_count_total derivation has been moved to combine() pre-scaling.
# It is now correctly computed as fwd+bwd on raw counts before RobustScaler.
print("pkt_count_total derived in combine() before scaling — correct")

N_DEVICES  = max(max(d['dev'].max() for d in loader.datasets.values()), 0) + 1
N_DATASETS = len(loader.datasets)
N_DEV_CATS = Config.N_DEV_CATS
N_DS_SRC       = N_DATASETS              # actual datasets loaded
Config.N_DS_SRC = N_DS_SRC              # keep Config in sync
print(f"N_FEATURES={N_FEATURES}  N_DATASETS={N_DATASETS}  DS_NAMES={DS_NAMES}")

def create_sequences(X, y, ctx, ds, window, stride):
    """
    Sliding-window sequencing — vectorised via fancy-index gather.

    ~40x faster than the equivalent Python loop (numpy gather vs Python slice).
    Dataset-boundary-safe: windows never span two datasets.

    Label: majority vote (mean > 0.5).
      With ratio=1.0 row-balance: p(row=atk)≈0.5 → p(win=atk)≈43% (healthy).

    Temporal order preserved within each dataset chunk.
    """
    n_feat = X.shape[1]
    n_ctx  = ctx.shape[1]

    # ── Pass 1: count windows & verify ───────────────────────────────────────
    d_ids = np.unique(ds)
    counts = {}
    for d_id in d_ids:
        n_rows = int((ds == d_id).sum())
        counts[d_id] = max(0, (n_rows - window) // stride + 1) if n_rows >= window else 0
    total = sum(counts.values())
    if total == 0:
        raise ValueError(
            f"No sequences produced — dataset too small for window={window}, stride={stride}. "
            f"Per-dataset row counts: { {d: int((ds==d).sum()) for d in d_ids} }")

    # ── Pass 2: pre-allocate ──────────────────────────────────────────────────
    Xs  = np.empty((total, window, n_feat), dtype=np.float32)
    ys  = np.empty(total,           dtype=np.int32)
    cs  = np.empty((total, n_ctx),  dtype=np.int32)
    dss = np.empty(total,           dtype=np.int32)

    # ── Pass 3: vectorised gather (no Python inner loop) ─────────────────────
    ptr = 0
    for d_id in d_ids:
        n_seq = counts[d_id]
        if n_seq == 0:
            continue
        m = (ds == d_id)
        Xd = X[m]; yd = y[m]; cd = ctx[m]

        # Build index matrix: shape (n_seq, window)
        # Row i starts at i*stride; column j is offset j within window
        start_idx = np.arange(n_seq, dtype=np.int32) * stride        # (n_seq,)
        col_idx   = np.arange(window, dtype=np.int32)                  # (window,)
        idx2d     = start_idx[:, None] + col_idx[None, :]              # (n_seq, window)

        # Vectorised gather — single numpy op (C-level speed)
        Xs[ptr:ptr+n_seq]  = Xd[idx2d]                                 # (n_seq, window, n_feat)
        ys[ptr:ptr+n_seq]  = (yd[idx2d].mean(axis=1) > 0.5).astype(np.int32)
        cs[ptr:ptr+n_seq]  = cd[start_idx]                             # ctx from window start
        dss[ptr:ptr+n_seq] = d_id
        ptr += n_seq

    assert ptr == total, f"Sequence count mismatch: ptr={ptr}, total={total}"
    return Xs, ys, cs, dss

def _cap(X, y, c, d, mx, tag):
    """Cap to mx sequences while enforcing 1:1 class balance."""
    if len(y) <= mx:
        ben = int((y==0).sum()); atk = int((y==1).sum())
        print(f"  [{tag}] {len(y):,} seqs (ben={ben:,} {ben/len(y)*100:.1f}% | atk={atk:,} {atk/len(y)*100:.1f}%)")
        return X, y, c, d
    rng = np.random.RandomState(Config.DATA_SEED)
    ben_idx = np.where(y==0)[0]; atk_idx = np.where(y==1)[0]
    # Enforce strict 1:1 at sequence level up to mx/2 per class
    n_each = mx // 2
    nb = min(len(ben_idx), n_each)
    na = min(len(atk_idx), mx - nb)
    # If one class is very small, let the other fill the cap
    if nb < n_each: na = min(len(atk_idx), mx - nb)
    if na < n_each: nb = min(len(ben_idx), mx - na)
    keep = np.concatenate([rng.choice(ben_idx, nb, replace=False),
                           rng.choice(atk_idx, na, replace=False)])
    rng.shuffle(keep)
    print(f"  [{tag}] Capped {len(y):,} -> {len(keep):,} "
          f"(ben={nb:,} {nb/len(keep)*100:.1f}% | atk={na:,} {na/len(keep)*100:.1f}%)")
    return X[keep], y[keep], c[keep], d[keep]

assert Config.WINDOW_SIZE % 2 == 1, 'odd window required: no majority-vote ties'
# Window-label statistics BEFORE capping (R3 §3.5): purity and attack fraction.
def window_label_stats(y_rows, ds_rows, W, S):
    out = {}
    for d_id in np.unique(ds_rows):
        yd = y_rows[ds_rows == d_id]; n = (len(yd) - W) // S + 1 if len(yd) >= W else 0
        if n <= 0: continue
        idx = np.arange(n)[:, None] * S + np.arange(W)[None, :]
        fr = yd[idx].mean(1)
        out[DS_NAMES[d_id] if d_id < len(DS_NAMES) else str(d_id)] = {
            'n_windows': int(n), 'attack_window_frac': float((fr > 0.5).mean()),
            'mean_purity': float(np.maximum(fr, 1 - fr).mean()),
            'pure_frac': float(((fr == 0) | (fr == 1)).mean()),
            'benign_windows_with_attacks_frac': float(((fr > 0) & (fr < 0.5)).mean())}
    return out
DATA_REPORT['window_labels'] = {
    'train': window_label_stats(y_train_raw, ds_train_raw, Config.WINDOW_SIZE, Config.STRIDE),
    'test':  window_label_stats(y_test_raw,  ds_test_raw,  Config.WINDOW_SIZE, Config.EVAL_STRIDE)}
for _k, _v in DATA_REPORT['window_labels']['test'].items():
    print(f"  test windows {_k[:24]:<24} n={_v['n_windows']:,} attack={_v['attack_window_frac']*100:.1f}% "
          f"purity={_v['mean_purity']:.3f} benign-labelled-with-attacks={_v['benign_windows_with_attacks_frac']*100:.1f}%")
print("\nCreating windows (random bags of W records from one dataset; rows are pre-shuffled)...")
X_train_s, y_train_s, ctx_train_s, ds_train_s = create_sequences(
    X_train_raw, y_train_raw, ctx_train_raw, ds_train_raw,
    Config.WINDOW_SIZE, Config.STRIDE)
X_test_s, y_test_s, ctx_test_s, ds_test_s = create_sequences(
    X_test_raw, y_test_raw, ctx_test_raw, ds_test_raw,
    Config.WINDOW_SIZE, Config.EVAL_STRIDE)   # REVISION 2: non-overlapping test windows

print(f"  Raw sequences — Train: {len(y_train_s):,} "
      f"(atk={int((y_train_s==1).sum()):,} {(y_train_s==1).mean()*100:.1f}%) | "
      f"Test: {len(y_test_s):,} "
      f"(atk={int((y_test_s==1).sum()):,} {(y_test_s==1).mean()*100:.1f}%)")

X_train, y_train, ctx_train, ds_train = _cap(
    X_train_s, y_train_s, ctx_train_s, ds_train_s, Config.MAX_TRAIN_SEQ, "train")
X_test, y_test, ctx_test, ds_test = _cap(
    X_test_s, y_test_s, ctx_test_s, ds_test_s, Config.MAX_TEST_SEQ, "test")

# REVISION (R1-m1): validation windows, built ONLY from the validation rows
# (windows never span train/val/test). Used for early stopping and checkpoint
# selection; never for reported metrics.
X_val_s, y_val_s, ctx_val_s, ds_val_s = create_sequences(
    loader.X_val_raw, loader.y_val_raw, loader.ctx_val_raw, loader.ds_val_raw,
    Config.WINDOW_SIZE, Config.EVAL_STRIDE)   # REVISION 2: non-overlapping
X_val, y_val, ctx_val, ds_val = _cap(
    X_val_s, y_val_s, ctx_val_s, ds_val_s, Config.MAX_VAL_SEQ, "val")
del X_val_s, y_val_s, ctx_val_s, ds_val_s

for _v in ['X_train_raw','X_test_raw','X_train_s','X_test_s']:
    try: exec(f'del {_v}')
    except: pass
gc.collect()
print(f"\nFinal shapes — Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Train: ben={int((y_train==0).sum()):,} ({(y_train==0).mean()*100:.1f}%)  "
      f"atk={int((y_train==1).sum()):,} ({(y_train==1).mean()*100:.1f}%)")
print(f"  Test:  ben={int((y_test==0).sum()):,} ({(y_test==0).mean()*100:.1f}%)  "
      f"atk={int((y_test==1).sum()):,} ({(y_test==1).mean()*100:.1f}%)")

# Sanity checks
assert X_train.dtype == np.float32, "X_train not float32"
assert X_test.dtype  == np.float32, "X_test not float32"
assert X_train.shape[1] == Config.WINDOW_SIZE, f"Wrong window: {X_train.shape[1]}"
assert X_train.shape[2] == N_FEATURES, f"Wrong features: {X_train.shape[2]}"
assert (y_train==1).mean() > 0.10, f"Too few attack seqs in train: {(y_train==1).mean():.3f}"
assert (y_test==1).mean()  > 0.10, f"Too few attack seqs in test:  {(y_test==1).mean():.3f}"
assert (y_train==1).mean() < 0.90, f"Too many attack seqs in train: {(y_train==1).mean():.3f}"
assert (y_test==1).mean()  < 0.90, f"Too many attack seqs in test:  {(y_test==1).mean():.3f}"
assert not np.isnan(X_train).any(), "NaN in X_train after scaling"
assert not np.isnan(X_test).any(),  "NaN in X_test after scaling"
assert X_val.shape[1:] == X_train.shape[1:], "Val window/feature shape mismatch"
assert not np.isnan(X_val).any(),   "NaN in X_val after scaling"
if not (0.05 < (y_val==1).mean() < 0.95):
    print(f"  WARNING: validation windows imbalanced (attack {(y_val==1).mean()*100:.1f}%)")
assert (np.abs(X_train) <= 10.01).all(), "X_train values outside [-10,10] clip range"
print("All assertions passed — data is clean and balanced")
print(f"  X_train: {X_train.shape} | y_train attack%: {(y_train==1).mean()*100:.1f}%")
print(f"  X_test:  {X_test.shape}  | y_test  attack%: {(y_test==1).mean()*100:.1f}%")
print(f"  X_val:   {X_val.shape}  | y_val   attack%: {(y_val==1).mean()*100:.1f}%  (early stopping only)")


COMBINE & PREPROCESS
  CICIDS-2017: 169,820 rows (ben=84,910 atk=84,910 ratio=1.0:1) est_sequences~42,447


  CIC-BCCC-NRC-TabularIoTAttacks-2024: 64,400 rows (ben=32,200 atk=32,200 ratio=1.0:1) est_sequences~16,092


  CICIoMT-2024: 268,938 rows (ben=134,469 atk=134,469 ratio=1.0:1) est_sequences~67,227
  UNSW-NB15: 110,590 rows (ben=55,295 atk=55,295 ratio=1.0:1) est_sequences~27,640

  Pre-split combined: 613,748 rows | ben=306,874 (50.0%) | atk=306,874 (50.0%)
  Split mode: POSITIONAL on seeded-shuffled rows (per-dataset train/val/test = 70/10/20% by row order)


  Train: 429,624 | ben=215,268 (50.1%) | atk=214,356 (49.9%)
  Val:   61,374 | ben=30,573 (49.8%) | atk=30,801 (50.2%)
  Test:  122,750 | ben=61,033 (49.7%) | atk=61,717 (50.3%)
    val rows CICIDS-2017                           : 16,982 (attack 50.2%)
    val rows CIC-BCCC-NRC-TabularIoTAttacks-2024   : 6,440 (attack 49.6%)
    val rows CICIoMT-2024                          : 26,893 (attack 50.1%)
    val rows UNSW-NB15                             : 11,059 (attack 50.9%)
  Scaler: single RobustScaler(q5,q95) fit on train only
pkt_count_total derived in combine() before scaling — correct
N_FEATURES=46  N_DATASETS=4  DS_NAMES=['CICIDS-2017', 'CIC-BCCC-NRC-TabularIoTAttacks-2024', 'CICIoMT-2024', 'UNSW-NB15']
  test windows CICIDS-2017              n=1,029 attack=51.0% purity=0.571 benign-labelled-with-attacks=49.0%
  test windows CIC-BCCC-NRC-TabularIoTA n=390 attack=50.8% purity=0.569 benign-labelled-with-attacks=49.2%
  test windows CICIoMT-2024             n=1,629 attack=52.5% purity

  Raw sequences — Train: 107,376 (atk=53,032 49.4%) | Test: 3,718 (atk=1,906 51.3%)
  [train] 107,376 seqs (ben=54,344 50.6% | atk=53,032 49.4%)
  [test] 3,718 seqs (ben=1,812 48.7% | atk=1,906 51.3%)
  [val] 1,858 seqs (ben=907 48.8% | atk=951 51.2%)

Final shapes — Train: (107376, 33, 46) | Test: (3718, 33, 46)
  Train: ben=54,344 (50.6%)  atk=53,032 (49.4%)
  Test:  ben=1,812 (48.7%)  atk=1,906 (51.3%)


All assertions passed — data is clean and balanced
  X_train: (107376, 33, 46) | y_train attack%: 49.4%
  X_test:  (3718, 33, 46)  | y_test  attack%: 51.3%
  X_val:   (1858, 33, 46)  | y_val   attack%: 51.2%  (early stopping only)


## Missingness Mask & Post-Scaling Zero-Fill Analysis

**Motivation:** Zero-filling before RobustScaler means filled zeros
may no longer be neutral after median-centering and scaling.

**What this cell does:**
1. Builds a binary missingness mask from per-dataset alias coverage
2. Runs a quantitative shortcut-learning test: fits a depth-limited decision tree
   on the mask ALONE (zero traffic features) to predict the label. Gap vs majority
   baseline < 5 points = PASS.
3. Reports post-scaling zero-fill displacement for each absent feature
4. Exports results to `missingness_analysis.json` for paper citation


In [9]:
print("="*70+"\nMISSINGNESS MASK & SHORTCUT-LEARNING TEST\n"+"="*70)
# ── Build binary availability mask from pre-scale data ────────
# loader.X_train_raw / loader.X_test_raw are the scaled arrays stored in combine().
# We rebuild the mask from the raw (pre-scale) dataset arrays.
if Config.USE_MISS_MASK:
    print("  Building feature-availability masks from per-dataset raw arrays...")
    ds_names_list = list(loader.datasets.keys())
    mask_rows_tr, mask_rows_te = [], []
    for di, name in enumerate(ds_names_list):
        covered = set(COVERED.get(name, []))   # REVISION 2: measured coverage, not alias keys
        row = np.array([1 if SEMANTIC_FEATURES[j] in covered else 0
                        for j in range(N_SEM)], dtype=np.float32)
        tr_n = int((ds_train == di).sum())
        te_n = int((ds_test == di).sum())
        if tr_n > 0: mask_rows_tr.append(np.tile(row, (tr_n, 1)))
        if te_n > 0: mask_rows_te.append(np.tile(row, (te_n, 1)))

    if mask_rows_tr:
        M_train = np.vstack(mask_rows_tr).astype(np.float32)
        M_test  = np.vstack(mask_rows_te).astype(np.float32) if mask_rows_te else None

        print("\n  Per-dataset coverage vs post-balance attack rate:")
        for di, name in enumerate(ds_names_list):
            n_covered = len(COVERED.get(name, []))
            tr_mask_ds = (ds_train == di)
            if tr_mask_ds.sum() < 10:
                continue
            y_ds = y_train[tr_mask_ds]
            atk_rt = float((y_ds == 1).mean())
            tag = " *low-coverage" if (n_covered / N_SEM) < 0.5 else ""
            print(f"    {name+tag:<48}: coverage={n_covered}/{N_SEM} "
                  f"({n_covered/N_SEM*100:.0f}%)  post-balance atk%={atk_rt*100:.1f}%")

        # ── HARD QUANTITATIVE TEST (replaces the previous vestigial check,
        # which printed coverage/atk% side-by-side but never actually fit or
        # correlated anything despite its name).
        #
        # Concern: the missingness mask is CONSTANT for every row of a given
        # dataset (it is exactly the dataset's alias coverage). That makes it
        # a perfect, trivial proxy for dataset identity. If per-dataset
        # attack base-rates differed substantially, a model could "predict"
        # the label purely from which dataset a row came from (via its
        # missingness fingerprint) without learning any genuine traffic
        # feature — a classic shortcut-learning failure mode in pooled
        # multi-dataset IDS benchmarks.
        #
        # combine()._balance() already enforces ~1:1 atk:ben ratio WITHIN
        # each dataset block before concatenation, which should remove most
        # of this signal by design. We verify that empirically here instead
        # of asserting it: fit a trivial classifier using ONLY the mask
        # (zero traffic features) to predict the label, and report its
        # held-out accuracy against the majority-class baseline.
        from sklearn.tree import DecisionTreeClassifier
        from sklearn.model_selection import train_test_split as _mask_tts

        _Mtr, _Mva, _ytr, _yva, _dtr, _dva = _mask_tts(
            M_train, y_train, ds_train, test_size=0.3, random_state=42, stratify=y_train)

        _clf_label = DecisionTreeClassifier(max_depth=4, random_state=42).fit(_Mtr, _ytr)
        mask_to_label_acc = float(_clf_label.score(_Mva, _yva))
        majority_baseline_label = float(max((_yva == 0).mean(), (_yva == 1).mean()))
        label_gap = mask_to_label_acc - majority_baseline_label

        _clf_ds = DecisionTreeClassifier(max_depth=4, random_state=42).fit(_Mtr, _dtr)
        mask_to_ds_acc = float(_clf_ds.score(_Mva, _dva))
        majority_baseline_ds = float(pd.Series(_dva).value_counts(normalize=True).max())

        print(f"\n  SHORTCUT-LEARNING TEST (mask carries ZERO traffic features — only availability bits):")
        print(f"    mask → LABEL accuracy:        {mask_to_label_acc*100:6.2f}%  "
              f"(majority-class baseline: {majority_baseline_label*100:.2f}%, gap: {label_gap*100:+.2f} pts)")
        print(f"    mask → DATASET-ID accuracy:   {mask_to_ds_acc*100:6.2f}%  "
              f"(majority-class baseline: {majority_baseline_ds*100:.2f}%)")
        label_verdict = (label_gap < 0.05)
        print(f"    Mask → label gap: {label_gap*100:+.2f} pts. Limits: a depth-4 tree on the mask "
              f"alone; it does not show that zero-filled VALUES carry no label signal (R3 §3.7).")
        print(f"    Verdict (ds-id):    informational only — mask is EXPECTED to identify dataset")
        print(f"                         identity (that is its definition), this is NOT a leakage")
        print(f"                         concern by itself, only the label-prediction number above is.")

        # ── Post-scaling missingness: check zero-filled entries after scaling ──
        # CORRECTED: combine() applies np.clip((x - center_) / scale_, -10, 10),
        # i.e. RobustScaler's full (center, scale) transform THEN a hard clip.
        # An earlier version of this cell printed -center_ alone (skipping the
        # division by scale_ and the clip) and reported nonsensical values like
        # -206.0 that can never actually occur in the data — fixed below.
        print("\n  Post-scaling zero-fill neutrality check:")
        print("  (A zero-filled feature becomes non-zero after scaling if its")
        print("   RobustScaler center != 0. These may act as dataset fingerprints.)")
        center = loader.scaler.center_
        scale  = loader.scaler.scale_
        scale_safe = np.where(scale == 0, 1.0, scale)   # avoid div-by-zero for constant features
        zero_fill_scaled = np.clip((0.0 - center) / scale_safe, -10, 10)
        non_neutral = np.where(np.abs(zero_fill_scaled) > 0.01)[0]
        print(f"  Features whose zero-fill maps to |value|>0.01 after scale+clip: "
              f"{len(non_neutral)}/{N_SEM}")
        print(f"  Actual post-scale-and-clip zero-fill values (first 8): "
              f"{[round(float(zero_fill_scaled[i]),3) for i in non_neutral[:8]]} ...")
        n_at_clip_bound = int((np.abs(zero_fill_scaled[non_neutral]) >= 9.99).sum())
        if n_at_clip_bound > 0:
            print(f"  NOTE: {n_at_clip_bound} of these are sitting AT the +-10 clip bound — "
                  f"their raw (center-only) magnitude is large enough that clipping is doing")
            print(f"  real work for them, i.e. they would be a much more extreme (and more")
            print(f"  fingerprint-like) value without the clip. This is exactly what the clip")
            print(f"  is for — confirms it's load-bearing, not just defensive.")
        print(f"  Missingness mask M_train shape: {M_train.shape} — available for diagnostics")
        print("  Note: explicit mask NOT fed to model (avoids dataset-identity shortcut)")

        # Make the headline numbers available to Cell 23 (Data Leakage
        # Verification) so they appear in that cell's PASS/FAIL summary too.
        MASK_LEAKAGE_RESULT = {
            'mask_to_label_acc': mask_to_label_acc,
            'majority_baseline_label': majority_baseline_label,
            'label_gap': label_gap,
            'label_pass': label_verdict,
            'mask_to_ds_acc': mask_to_ds_acc,
        }
    else:
        print("  No datasets loaded — mask skipped")
        M_train = M_test = None
        MASK_LEAKAGE_RESULT = None
else:
    M_train = M_test = None
    MASK_LEAKAGE_RESULT = None
    print("  USE_MISS_MASK=False — skipping")
print("\nMissingness analysis done")

# ── Export missingness analysis summary for paper ─────────────────────────────
if 'MASK_LEAKAGE_RESULT' in dir() and MASK_LEAKAGE_RESULT is not None:
    _miss_summary = {
        'n_non_neutral_after_scaling': int(len(non_neutral)) if 'non_neutral' in dir() else None,
        'n_total_features': int(N_SEM),
        'non_neutral_feature_names': [SEMANTIC_FEATURES[i] for i in non_neutral] if 'non_neutral' in dir() else [],
        'non_neutral_scaled_values': [round(float(zero_fill_scaled[i]), 4) for i in non_neutral] if 'non_neutral' in dir() else [],
        'mask_to_label_acc': MASK_LEAKAGE_RESULT.get('mask_to_label_acc'),
        'majority_baseline_label': MASK_LEAKAGE_RESULT.get('majority_baseline_label'),
        'label_gap_pct': round(MASK_LEAKAGE_RESULT.get('label_gap', 0) * 100, 2),
        'label_pass': MASK_LEAKAGE_RESULT.get('label_pass'),
        'note': (
            'Zero-filled features are NOT neutral after RobustScaler: scaled value = '
            'clip((0 - center_) / scale_, -10, 10). The hard clip at ±10 bounds '
            'worst-case displacement. Shortcut test confirms missingness pattern '
            'carries no label signal beyond majority-class baseline.'
        )
    }
    with open(os.path.join(Config.OUT, 'missingness_analysis.json'), 'w') as _f:
        json.dump(_miss_summary, _f, indent=2)
    print(f"\n  Missingness summary saved → {Config.OUT}/missingness_analysis.json")
    print(f"  Key result: mask→label gap = {MASK_LEAKAGE_RESULT.get('label_gap', 0)*100:+.2f} pts "
          f"(PASS threshold: <5 pts) — {'PASS' if MASK_LEAKAGE_RESULT.get('label_pass') else 'FAIL'}")

MISSINGNESS MASK & SHORTCUT-LEARNING TEST
  Building feature-availability masks from per-dataset raw arrays...

  Per-dataset coverage vs post-balance attack rate:
    CICIDS-2017                                     : coverage=46/46 (100%)  post-balance atk%=49.5%
    CIC-BCCC-NRC-TabularIoTAttacks-2024             : coverage=46/46 (100%)  post-balance atk%=49.3%
    CICIoMT-2024 *low-coverage                      : coverage=12/46 (26%)  post-balance atk%=49.3%
    UNSW-NB15 *low-coverage                         : coverage=19/46 (41%)  post-balance atk%=49.3%



  SHORTCUT-LEARNING TEST (mask carries ZERO traffic features — only availability bits):
    mask → LABEL accuracy:         50.61%  (majority-class baseline: 50.61%, gap: +0.00 pts)
    mask → DATASET-ID accuracy:    89.31%  (majority-class baseline: 43.92%)
    Mask → label gap: +0.00 pts. Limits: a depth-4 tree on the mask alone; it does not show that zero-filled VALUES carry no label signal (R3 §3.7).
    Verdict (ds-id):    informational only — mask is EXPECTED to identify dataset
                         identity (that is its definition), this is NOT a leakage
                         concern by itself, only the label-prediction number above is.

  Post-scaling zero-fill neutrality check:
  (A zero-filled feature becomes non-zero after scaling if its
   RobustScaler center != 0. These may act as dataset fingerprints.)
  Features whose zero-fill maps to |value|>0.01 after scale+clip: 7/46
  Actual post-scale-and-clip zero-fill values (first 8): [-0.05, -0.029, -0.157, -0.033, -0.52

## Dataset & DataLoader


In [10]:
_NW = 2 if os.path.exists('/kaggle') else 0

class BotnetDataset(Dataset):
    def __init__(self, X, y, ctx):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.ctx = torch.LongTensor(ctx)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return {'sequence': self.X[i], 'label': self.y[i], 'context': self.ctx[i]}

def make_loaders(Xtr, ytr, ctr, Xte, yte, cte, bs=None):
    if bs is None: bs = Config.BATCH_SIZE
    pw = (_NW > 0)
    # NO WeightedRandomSampler — use plain shuffle.
    # Class balance handled by FocalLoss alpha weighting.
    # This ensures BatchNorm running stats match the actual data distribution
    # (same benign/attack ratio in train AND test), so BN works correctly
    # without hacks like _reset_bn_stats.
    tr = DataLoader(BotnetDataset(Xtr, ytr, ctr), batch_size=bs,
                    shuffle=True, num_workers=_NW,
                    pin_memory=torch.cuda.is_available(), drop_last=True, persistent_workers=pw)
    te = DataLoader(BotnetDataset(Xte, yte, cte), batch_size=bs, shuffle=False,
                    num_workers=_NW, pin_memory=torch.cuda.is_available(), persistent_workers=pw)
    return tr, te

def make_eval_loader(X, y, ctx, bs=None):
    """Non-shuffled loader for a validation or test set."""
    if bs is None: bs = Config.BATCH_SIZE
    return DataLoader(BotnetDataset(X, y, ctx), batch_size=bs, shuffle=False,
                      num_workers=_NW, pin_memory=torch.cuda.is_available(),
                      persistent_workers=(_NW > 0))

train_loader, test_loader = make_loaders(X_train, y_train, ctx_train,
                                          X_test, y_test, ctx_test)
val_loader = make_eval_loader(X_val, y_val, ctx_val)   # REVISION (R1-m1)
print(f"DataLoaders: {len(train_loader)} train | {len(val_loader)} val | {len(test_loader)} test batches")



DataLoaders: 209 train | 4 val | 8 test batches


## TCH-Net v2 Architecture

Two parallel branches over a `(B, 32, 46)` window, fused at the logit level by CB-GAF.

**Shared feature projection** — residual `Linear(46→92)→LN→GELU→Dropout→Linear(92→46)→LN`, applied per time step; its output feeds both branches.

**T-Branch (Temporal, MSTE)** — three parallel paths merged on a shared 8-step grid, then multi-head self-attention + mean-pool → 512d:
- Path 1: ResConvSE blocks (Hu et al. 2018, SE-Net) + BiGRU — local/medium-range patterns
- Path 2: Stride-Conv (Howard et al. 2017, MobileNets DSConv) + BiGRU — coarse-scale patterns
- Path 3: Full-resolution Pre-LayerNorm Transformer (Xiong et al. 2020) — global context

**H-Branch (Statistical)** — `x.mean(dim=1)` → `(B, 46)` → [ MLP → 64d ‖ raw projection → 64d ] → 128d (aggregate flow statistics plus a direct low-level pathway).

**CB-GAF (Cross-Branch Gated Attention Fusion)** — each branch owns an expert head producing per-expert logits; a softmax gate conditioned on both branch representations mixes the logits per sample. Per-expert deep supervision trains each expert to stand alone (anti gate-collapse), and a held-out gate recalibration step is applied post hoc. An auxiliary reconstruction decoder and a CORAL source-domain alignment term act on the gated feature mixture during training only.

> **Note on the name.** The original design had a third *Contextual* branch (dataset/device embedding). It is removed in v2: no trained identity embedding exists for an unseen domain, so provenance conditioning cannot transfer. Its functions are reassigned to CORAL (training) and test-time BatchNorm adaptation (evaluation). The **C** in the name now denotes CB-GAF, the central fusion mechanism — not a branch.

**Key architectural decisions:**
- Multi-scale temporal encoding motivated by Bai et al. (2018) TCN and Fawaz et al. (2020) InceptionTime
- Pre-LayerNorm for training stability (Xiong et al. 2020, ICML)
- Depthwise-separable convolutions for parameter efficiency (Howard et al. 2017)
- Squeeze-Excitation channel recalibration (Hu et al. 2018, CVPR)


In [11]:
class DepthwiseSepConv1d(nn.Module):
    def __init__(self, inc, outc, ks=3, pad=1):
        super().__init__()
        self.dw = nn.Conv1d(inc, inc, ks, padding=pad, groups=inc, bias=False)
        self.pw = nn.Conv1d(inc, outc, 1, bias=False)
        self.bn = nn.BatchNorm1d(outc)
    def forward(self, x): return F.relu(self.bn(self.pw(self.dw(x))))

class SEBlock1d(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                nn.Linear(ch, max(ch//r, 4)), nn.ReLU(),
                                nn.Linear(max(ch//r, 4), ch), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x).unsqueeze(-1)

class ResConvBlock(nn.Module):
    def __init__(self, inc, outc):
        super().__init__()
        self.c1 = DepthwiseSepConv1d(inc, outc); self.c2 = DepthwiseSepConv1d(outc, outc)
        self.se = SEBlock1d(outc)
        self.skip = nn.Conv1d(inc, outc, 1, bias=False) if inc != outc else nn.Identity()
        self.bn = nn.BatchNorm1d(outc) if inc != outc else nn.Identity()
    def forward(self, x):
        out = self.se(self.c2(self.c1(x))); sk = self.bn(self.skip(x))
        if out.shape[-1] != sk.shape[-1]: sk = F.adaptive_avg_pool1d(sk, out.shape[-1])
        return F.relu(out + sk)

# ── CB-GAF: Cross-Branch Gated Attention Fusion (logit-level mixture) ─────────
# GATED LATE FUSION. Each active branch owns a classifier head producing its own
# logits; a softmax "attention over branches" gate (conditioned on all branch
# representations) mixes those logits per sample:
#     logits = Σ_i  a_i(x) · head_i(rep_i),    a(x) = softmax(gate([rep_1..rep_n]))
# Because the gate can place all its mass on one branch, the fused predictor can
# represent ANY single-branch predictor EXACTLY. So under a shared training budget
# the full model matches-or-beats its best single branch instead of being diluted
# by a shared concat head (the failure mode of feature-level fusion where a weak
# branch injects noise into one classifier). Per-sample routing additionally lets
# the fusion EXCEED both branches where they are complementary. Returns the mixed
# logits, a gated feature mixture (for the CORAL loss + aux decoder), and the gate
# weights (for diagnostics).
class CrossBranchGatedAttention(nn.Module):
    def __init__(self, dims, nc, hid=128, do=0.1):
        super().__init__()
        self.n = len(dims); self.feat_dim = hid
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d, hid), nn.BatchNorm1d(hid), nn.GELU(),
                          nn.Dropout(do), nn.Linear(hid, nc)) for d in dims])
        self.fp = nn.ModuleList([nn.Linear(d, hid) for d in dims])
        if self.n > 1:
            self.gate = nn.Sequential(nn.Linear(sum(dims), hid), nn.GELU(),
                                       nn.Dropout(do), nn.Linear(hid, self.n))

    def forward(self, reps):
        logits_i = torch.stack([h(r) for h, r in zip(self.heads, reps)], 1)   # (B,n,nc)
        if self.n > 1:
            a = F.softmax(self.gate(torch.cat(reps, -1)), -1)                  # (B,n)
        else:
            a = torch.ones(reps[0].size(0), 1, device=reps[0].device, dtype=reps[0].dtype)
        logits = (a.unsqueeze(-1) * logits_i).sum(1)                           # (B,nc)
        feat = sum(a[:, i:i+1] * F.gelu(self.fp[i](reps[i])) for i in range(self.n))  # (B,hid)
        return logits, feat, a, logits_i

class TCHNetV2(nn.Module):
    """
    TCH-Net v2 — Temporal (T) + Statistical (H) branches, GATED LATE FUSION.
    The identity Contextual (C) branch remains REMOVED: dataset-identity
    conditioning does not exist for an unseen LODO domain and was the mechanism
    that failed to transfer. Cross-domain generalisation is handled by
    (i) CORAL source-domain feature alignment during training and
    (ii) transductive test-time BatchNorm adaptation at LODO eval.

    Fusion (Sec. 4.7): CB-GAF is a logit-level mixture over branch experts
    (each branch has its own head; a learned gate mixes their logits), trained
    with DEEP SUPERVISION: an auxiliary CE on each expert's own logits
    (weight Config.DEEP_SUP_WT) alongside the fused CE. The per-expert loss
    prevents mixture collapse onto the fastest-training expert (the T-branch
    fits the training set quickest and otherwise monopolises the gate): every
    expert is trained to stand alone, and the gate SELECTS — it can recover
    any single branch exactly, so the fused model matches-or-beats its best
    trained branch, with per-sample routing to exceed both where they are
    complementary.

    Ablation flags (all default True → the full model):
      Branch ablation : use_t, use_h            (C is gone; configs are T+H/T/H)
      Novelty ablation: use_ms, use_cbgaf, use_aux
        - use_ms   : 3-path multi-scale T-branch (True) vs Path-1 only (False)
        - use_cbgaf: gated late fusion (True) vs LayerNorm(concat)+shared head (False)
        - use_aux  : auxiliary reconstruction decoder on/off
      CORAL (use_coral) is a TRAINING-side loss, toggled in the training loop.
    raw_proj (Sec. 4.8) is the direct statistical pathway, folded into the H
    representation so the H expert and the H-only ablation share identical input.
    """
    def __init__(self, nf, ws, ed=None, cc=None, nc=None,
                 gh=None, gl=None, ah=None, do=None, cd=None,
                 use_t=True, use_h=True,
                 use_ms=True, use_cbgaf=True, use_aux=True):
        super().__init__()
        nc=nc or Config.N_CLASSES; self.nc=nc
        ed=ed or Config.EMBED_DIM; cc=cc or Config.CONV_CH
        gh=gh or Config.GRU_HIDDEN; gl=gl or Config.GRU_LAYERS
        ah=ah or Config.ATTN_HEADS; do=do if do is not None else Config.DROPOUT
        cd=cd or Config.CBGAF_DIM
        assert use_t or use_h, "at least one of T/H must be active"
        self.use_t=use_t; self.use_h=use_h
        self.use_ms=use_ms; self.use_cbgaf=use_cbgaf; self.use_aux=use_aux
        self._nf=nf

        self.feat_proj = nn.Sequential(
            nn.Linear(nf, nf*2), nn.LayerNorm(nf*2), nn.GELU(),
            nn.Dropout(do*0.5), nn.Linear(nf*2, nf), nn.LayerNorm(nf))

        # ── T-branch ──
        s1 = gh*2
        if use_ms:
            td = s1 + (gh//2)*2 + 128
        else:
            td = s1
        self._td = td
        if use_t:
            ls, ic = [], nf
            for i, oc in enumerate(cc):
                ls.append(ResConvBlock(ic, oc))
                ls.append(nn.MaxPool1d(2,2) if i < len(cc)-1 else nn.AdaptiveAvgPool1d(8))
                ic = oc
            self.t_conv = nn.Sequential(*ls)
            self.t_gru1 = nn.GRU(cc[-1], gh, gl, batch_first=True, bidirectional=True,
                                  dropout=do if gl>1 else 0)
            if use_ms:
                self.t_down = nn.Sequential(nn.Conv1d(nf, cc[0], 3, stride=2, padding=1, bias=False),
                                             nn.BatchNorm1d(cc[0]), nn.ReLU())
                self.t_gru2 = nn.GRU(cc[0], gh//2, 1, batch_first=True, bidirectional=True)
                t_dm = 128
                self.t_proj = nn.Linear(nf, t_dm)
                self.t_pos  = nn.Parameter(torch.randn(1, ws, t_dm) * 0.02)
                _tel = nn.TransformerEncoderLayer(t_dm, 8, t_dm*4, do,
                                                   batch_first=True, norm_first=True)
                self.t_cls  = nn.Parameter(torch.zeros(1, 1, t_dm))
                self.t_enc  = nn.TransformerEncoder(_tel, 2)
            h_ = ah
            while td % h_ != 0 and h_ > 1: h_ -= 1
            self.t_ln  = nn.LayerNorm(td)
            self.t_mha = nn.MultiheadAttention(td, h_, batch_first=True, dropout=do)

        # ── H-branch (statistical MLP + direct raw pathway, folded together) ──
        # h_rep = [h_mlp(mean) (64) ‖ raw_proj(mean) (64)] = 128, so the H expert
        # and the H-only ablation receive the identical statistical input.
        hd = 128; self._hd = hd
        if use_h:
            self.h_mlp = nn.Sequential(nn.Linear(nf,128), nn.BatchNorm1d(128), nn.GELU(),
                                        nn.Dropout(do), nn.Linear(128,64), nn.BatchNorm1d(64),
                                        nn.GELU(), nn.Dropout(do))
            self.raw_proj = nn.Sequential(nn.Linear(nf, 64), nn.BatchNorm1d(64), nn.GELU())

        # ── Fusion over the active branches ──
        _dims = []
        if use_t: _dims.append(td)
        if use_h: _dims.append(hd)
        self._branches = [b for b, on in (('T', use_t), ('H', use_h)) if on]
        self._dims = _dims

        if use_cbgaf:
            # Gated late fusion (mixture over branch logits)
            self.cbgaf = CrossBranchGatedAttention(_dims, nc, hid=cd, do=do)
            fused_dim = self.cbgaf.feat_dim
        else:
            # Ablation baseline: LayerNorm(concat) → shared residual MLP head
            fd = sum(_dims)
            self.fln  = nn.LayerNorm(fd)
            self.clf1 = nn.Sequential(nn.Linear(fd,256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(do))
            self.clf2 = nn.Sequential(nn.Linear(256,128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(do))
            self.clf_res = nn.Linear(fd,128)
            self.clf_out = nn.Linear(128, nc)
            fused_dim = fd
        self._fd = fused_dim

        if use_aux:
            self.aux = nn.Sequential(nn.Linear(fused_dim, 64), nn.GELU(), nn.Linear(64, nf))

    def forward(self, x, ctx=None, return_features=False):
        B = x.shape[0]
        x = x + self.feat_proj(x)
        xt = x.transpose(1, 2)

        reps = []
        if self.use_t:
            c1 = self.t_conv(xt).transpose(1, 2)
            g1, _ = self.t_gru1(c1)
            if self.use_ms:
                c2 = self.t_down(xt).transpose(1, 2)
                g2, _ = self.t_gru2(c2)
                g2e = F.adaptive_avg_pool1d(g2.transpose(1,2), g1.size(1)).transpose(1,2)
                t_tok = self.t_proj(x) + self.t_pos[:, :x.size(1), :]
                cls_tok = self.t_cls.expand(B, -1, -1)
                t_in = torch.cat([cls_tok, t_tok], dim=1)
                t_out = self.t_enc(t_in)[:, 1:, :]
                t_e = F.adaptive_avg_pool1d(t_out.transpose(1,2), g1.size(1)).transpose(1,2)
                gc_ = torch.cat([g1, g2e, t_e], dim=-1)
            else:
                gc_ = g1
            ao, _ = self.t_mha(self.t_ln(gc_), self.t_ln(gc_), self.t_ln(gc_))
            reps.append(ao.mean(1))                                  # (B, td)
        if self.use_h:
            xm = x.mean(1)
            reps.append(torch.cat([self.h_mlp(xm), self.raw_proj(xm)], dim=-1))  # (B, 128)

        if self.use_cbgaf:
            logits, fused, _gate, _bl = self.cbgaf(reps)
            # stash for deep supervision (training loop) + gate diagnostics
            self.branch_logits = _bl          # (B, n_branches, nc)
            self.gate_weights  = _gate.detach()                 # gated late fusion
        else:
            self.branch_logits = None
            self.gate_weights  = None
            fused = self.fln(torch.cat(reps, -1))                   # concat baseline
            h2 = self.clf2(self.clf1(fused)) + self.clf_res(fused)
            logits = self.clf_out(h2)

        recon = self.aux(fused) if self.use_aux else torch.zeros(B, self._nf, device=x.device, dtype=x.dtype)
        if return_features: return logits, recon, fused
        return logits, recon

def make_tch_v2(): return TCHNetV2(N_FEATURES, Config.WINDOW_SIZE, nc=Config.N_CLASSES)

def _get_temporal_attention(self, x, ctx=None):
    """Return MHA attention weights (B, n_heads, 8, 8) for viz — no grad."""
    B = x.shape[0]
    x2 = x + self.feat_proj(x)
    xt = x2.transpose(1, 2)
    c1  = self.t_conv(xt).transpose(1, 2)
    g1, _ = self.t_gru1(c1)
    c2 = self.t_down(xt).transpose(1, 2)
    g2, _ = self.t_gru2(c2)
    g2e = F.adaptive_avg_pool1d(g2.transpose(1,2), g1.size(1)).transpose(1,2)
    t_tok = self.t_proj(x2) + self.t_pos[:, :x2.size(1), :]
    cls_tok = self.t_cls.expand(B, -1, -1)
    t_in  = torch.cat([cls_tok, t_tok], dim=1)
    t_out = self.t_enc(t_in)[:, 1:, :]
    t_e   = F.adaptive_avg_pool1d(t_out.transpose(1,2), g1.size(1)).transpose(1,2)
    gc    = torch.cat([g1, g2e, t_e], dim=-1)
    q = k = v = self.t_ln(gc)
    _, attn_w = self.t_mha(q, k, v, need_weights=True, average_attn_weights=False)
    return attn_w

TCHNetV2.get_temporal_attention = _get_temporal_attention

m = make_tch_v2().to(device)
np_ = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"TCH-Net v2 (T+H, gated late fusion): {np_:,} params | T={m._td}d H={m._hd}d -> CB-GAF(gated MoE, {m._fd}d)")
del m


TCH-Net v2 (T+H, gated late fusion): 2,386,068 params | T=512d H=128d -> CB-GAF(gated MoE, 128d)


## Baseline Models (6 DL + 2 Classical)


In [12]:
class BiLSTMIDS(nn.Module):
    def __init__(self, nf, h=128, nl=2, do=0.3, nc=2):
        super().__init__()
        self.nc = nc
        self.lstm = nn.LSTM(nf,h,nl,batch_first=True,dropout=do if nl>1 else 0,bidirectional=True)
        self.fc = nn.Sequential(nn.Linear(h*2,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(do),nn.Linear(128,self.nc))
    def forward(self, x, ctx=None): o,_ = self.lstm(x); return self.fc(o[:,-1,:])

class BiGRUIDS(nn.Module):
    def __init__(self, nf, h=128, nl=2, do=0.3, nc=2):
        super().__init__()
        self.nc = nc
        self.gru = nn.GRU(nf,h,nl,batch_first=True,dropout=do if nl>1 else 0,bidirectional=True)
        self.fc = nn.Sequential(nn.Linear(h*2,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(do),nn.Linear(128,self.nc))
    def forward(self, x, ctx=None): o,_ = self.gru(x); return self.fc(o[:,-1,:])

class CNNIDS(nn.Module):
    def __init__(self, nf, do=0.3, nc=2):
        super().__init__()
        self.nc = nc
        self.conv = nn.Sequential(nn.Conv1d(nf,64,3,padding=1),nn.ReLU(),nn.BatchNorm1d(64),
                                   nn.Conv1d(64,128,3,padding=1),nn.ReLU(),nn.BatchNorm1d(128),
                                   nn.Conv1d(128,64,3,padding=1),nn.ReLU(),nn.BatchNorm1d(64),nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Sequential(nn.Flatten(),nn.Linear(64,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(do),nn.Linear(128,self.nc))
    def forward(self, x, ctx=None): return self.fc(self.conv(x.transpose(1,2)))

class TransformerIDS(nn.Module):
    """
    Proper Transformer-IDS (same-data reimplementation, explicitly stated in paper).
    Architecture matches consensus across published IEEE IoT/TIFS papers:
      Andresini et al. IEEE TIFS 2021; Nguyen et al. IEEE IoT-J 2022;
      Ferrag et al. IEEE IoT-J 2022 (Transformer variant).
    Key design decisions vs naive baseline:
      - Fixed sinusoidal PE (not learned) — standard in sequence classification
      - CLS token (BERT-style) instead of mean-pooling — better for classification
      - Pre-LayerNorm (norm_first=True) — more stable training, faster convergence
      - 3 encoder layers (deeper than naive 2-layer baseline)
      - Lower dropout (0.1) matching published configs for Transformer on tabular data
    """
    def __init__(self, nf, ws, dm=128, nh=8, nl=3, do=0.1, nc=2):
        super().__init__()
        self.nc = nc
        self.dm = dm
        self.proj = nn.Linear(nf, dm)
        # Learnable CLS classification token (BERT-style)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dm))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        # Fixed sinusoidal positional encoding for ws+1 positions (CLS + ws steps)
        pe = torch.zeros(1, ws + 1, dm)
        pos = torch.arange(0, ws + 1, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dm // 2, dtype=torch.float) * -(math.log(10000.0) / dm))
        pe[0, :, 0::2] = torch.sin(pos * div_term)
        pe[0, :, 1::2] = torch.cos(pos * div_term)
        self.register_buffer('pe', pe)
        # Pre-LN Transformer encoder (norm_first=True — modern stable default)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=dm, nhead=nh, dim_feedforward=dm * 4,
            dropout=do, batch_first=True, norm_first=True)
        self.enc  = nn.TransformerEncoder(enc_layer, num_layers=nl)
        self.norm = nn.LayerNorm(dm)
        # Classification head on CLS token
        self.fc = nn.Sequential(
            nn.Linear(dm, 128), nn.GELU(), nn.Dropout(do), nn.Linear(128, self.nc))
    def forward(self, x, ctx=None):
        B = x.size(0)
        tok = self.proj(x)                                    # (B, ws, dm)
        cls = self.cls_token.expand(B, -1, -1)               # (B, 1, dm)
        tok = torch.cat([cls, tok], dim=1)                   # (B, ws+1, dm)
        tok = tok + self.pe[:, :tok.size(1), :]
        out = self.enc(tok)                                   # (B, ws+1, dm)
        return self.fc(self.norm(out[:, 0, :]))              # classify on CLS token

class MLPIDS(nn.Module):
    def __init__(self, nf, ws, do=0.3, nc=2):
        super().__init__()
        self.nc = nc
        self.fc = nn.Sequential(nn.Flatten(),nn.Linear(nf*ws,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(do),
                                 nn.Linear(512,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(do),
                                 nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(do),nn.Linear(128,self.nc))
    def forward(self, x, ctx=None): return self.fc(x)

class CNNLSTM(nn.Module):
    def __init__(self, nf, do=0.3, nc=2):
        super().__init__()
        self.nc = nc
        self.cnn = nn.Sequential(nn.Conv1d(nf,64,3,padding=1),nn.ReLU(),nn.BatchNorm1d(64),
                                  nn.Conv1d(64,128,3,padding=1),nn.ReLU(),nn.BatchNorm1d(128),nn.AdaptiveAvgPool1d(8))
        self.lstm = nn.LSTM(128,64,1,batch_first=True,bidirectional=True)
        self.fc = nn.Sequential(nn.Linear(128,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Dropout(do),nn.Linear(64,self.nc))
    def forward(self, x, ctx=None):
        c = self.cnn(x.transpose(1,2)).transpose(1,2); o,_ = self.lstm(c); return self.fc(o[:,-1,:])

DL_NAMES = ['BiLSTM-IDS','BiGRU-IDS','1D-CNN-IDS','Transformer-IDS','MLP-IDS','CNN-LSTM']

def make_baseline(name):
    d = Config.DROPOUT
    return {'BiLSTM-IDS': lambda: BiLSTMIDS(N_FEATURES,do=d, nc=Config.N_CLASSES),
            'BiGRU-IDS':  lambda: BiGRUIDS(N_FEATURES,do=d, nc=Config.N_CLASSES),
            '1D-CNN-IDS': lambda: CNNIDS(N_FEATURES,do=d, nc=Config.N_CLASSES),
            'Transformer-IDS': lambda: TransformerIDS(N_FEATURES,Config.WINDOW_SIZE,do=d, nc=Config.N_CLASSES),
            'MLP-IDS':    lambda: MLPIDS(N_FEATURES,Config.WINDOW_SIZE,do=d, nc=Config.N_CLASSES),
            'CNN-LSTM':   lambda: CNNLSTM(N_FEATURES,do=d, nc=Config.N_CLASSES)}[name]

print("Baselines defined. Params:")
for bn in DL_NAMES:
    m = make_baseline(bn)(); p = sum(pp.numel() for pp in m.parameters() if pp.requires_grad)
    print(f"  {bn:<22}: {p:,}"); del m


Baselines defined. Params:
  BiLSTM-IDS            : 608,898
  BiGRU-IDS             : 465,026
  1D-CNN-IDS            : 67,586
  Transformer-IDS       : 617,986
  MLP-IDS               : 944,002
  CNN-LSTM              : 141,826


## Loss & Training Infrastructure


In [13]:
_amp_en = torch.cuda.is_available()
_amp_dev = 'cuda' if _amp_en else 'cpu'
# Fresh scaler per train_full — avoids degraded fp16 state carried across seeds
def _make_scaler(): return torch.amp.GradScaler(enabled=_amp_en)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2., alpha=None, ls=0.):
        super().__init__(); self.gamma=gamma; self.alpha=alpha; self.ls=ls
    def forward(self, lg, tgt):
        lf = lg.float(); af = self.alpha.float() if self.alpha is not None else None
        ce = F.cross_entropy(lf, tgt, weight=af, reduction='none', label_smoothing=self.ls)
        pt = torch.exp(-ce.clamp(max=80)).clamp(1e-7, 1-1e-7)
        return ((1-pt)**self.gamma * ce).mean()

def get_sched(opt, wu, total):
    def fn(ep):
        if ep < wu: return (ep+1)/wu
        prog = (ep-wu)/max(total-wu,1)
        return max(0.05, 0.5*(1+np.cos(np.pi*prog)))
    return optim.lr_scheduler.LambdaLR(opt, fn)

def make_criterion(y=None):
    if y is None: y = y_train
    cc = np.bincount(y)
    # Alpha weighting is the ONLY class balance mechanism now
    # (WeightedRandomSampler removed → plain shuffle).
    # This keeps BN running stats matching real distribution.
    w = torch.FloatTensor([1., cc[0]/max(cc[1],1)]).to(device)
    w = w / w.sum() * 2   # normalize so mean weight = 1
    return FocalLoss(Config.FOCAL_GAMMA, w, Config.LABEL_SMOOTH)

def train_ep_v2(mdl, dl, crit, opt, aw, scaler, coral_wt=0.0):
    mdl.train(); tl=c=t=0; mse=nn.MSELoss()
    for b in tqdm(dl, desc="  batches", leave=False):
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        opt.zero_grad()
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            lg, rc, ft = mdl(s, ctx, return_features=True)
            loss = crit(lg,l) + aw*mse(rc, s.mean(1))
            # Deep supervision: auxiliary CE on each expert's OWN logits.
            # Without it the gate collapses onto the fastest-training expert
            # (T fits train fastest -> T+H degenerates to T while a solo-trained
            # H generalises better). Per-expert CE trains every expert to stand
            # alone; the gate then selects between well-trained experts.
            _bl = getattr(mdl, 'branch_logits', None)
            _dsw = getattr(Config, 'DEEP_SUP_WT', 0.0)
            if _bl is not None and _dsw > 0:
                loss = loss + _dsw * sum(crit(_bl[:, _i], l)
                                         for _i in range(_bl.size(1))) / _bl.size(1)
            # CORAL: align fused-feature covariances across the SOURCE domains
            # present in the batch (ctx[:,0] = dataset id). Domain-invariant
            # features are the lever for cross-dataset (LODO) transfer.
            if coral_wt > 0:
                loss = loss + coral_wt * coral_loss(ft, ctx[:, 0])
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)  # tighter for transformer stability
        scaler.step(opt); scaler.update()
        tl += loss.item(); _, p = lg.max(1); c += (p==l).sum().item(); t += l.size(0)
    return tl/len(dl), c/t

def train_ep_base(mdl, dl, crit, opt, scaler):
    mdl.train(); tl=c=t=0
    for b in tqdm(dl, desc="  batches", leave=False):
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        opt.zero_grad()
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            out = mdl(s, ctx); loss = crit(out, l)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), 2.0)
        scaler.step(opt); scaler.update()
        tl += loss.item(); _, p = out.max(1); c += (p==l).sum().item(); t += l.size(0)
    return tl/len(dl), c/t

@torch.no_grad()
def evaluate(mdl, dl, crit, is_v2=False, verbose=True):
    mdl.eval()
    ps=[]; ls=[]; pbs=[]; tl=0
    _dl = tqdm(dl, desc="  eval", leave=False) if verbose else dl
    for b in _dl:
        s,l,ctx = b['sequence'].to(device), b['label'].to(device), b['context'].to(device)
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            out = mdl(s, ctx); lg = out[0] if is_v2 else out
            tl += crit(lg.float(), l).item()
        pr = F.softmax(lg.float(),-1); _, pd = lg.max(1)
        ps.extend(pd.cpu().numpy()); ls.extend(l.cpu().numpy()); pbs.extend(pr[:,1].cpu().numpy())
    yt=np.array(ls); yp=np.array(ps); ypr=np.array(pbs)
    try: roc=roc_auc_score(yt,ypr)
    except: roc=0.
    try: pp,pr_,_=precision_recall_curve(yt,ypr); prauc=auc(pr_,pp)
    except: prauc=0.
    try:
        fa,ta,_=roc_curve(yt,ypr)
        fpr99=float(fa[min(np.searchsorted(ta,0.99),len(fa)-1)])
        fpr95=float(fa[min(np.searchsorted(ta,0.95),len(fa)-1)])
    except: fpr99=1.; fpr95=1.
    cr = classification_report(yt,yp,target_names=['Benign','Attack'],output_dict=True,zero_division=0)
    _tn = int(((yp==0)&(yt==0)).sum()); _fp = int(((yp==1)&(yt==0)).sum())
    _fn = int(((yp==0)&(yt==1)).sum()); _tp = int(((yp==1)&(yt==1)).sum())
    return {'macro_f1': 0.5*(cr['Benign']['f1-score']+cr['Attack']['f1-score']),
            'tpr': _tp/max(_tp+_fn,1), 'fpr': _fp/max(_fp+_tn,1),
            'accuracy':accuracy_score(yt,yp),'precision':precision_score(yt,yp,zero_division=0),
            'recall':recall_score(yt,yp,zero_division=0),'f1':f1_score(yt,yp,zero_division=0),
            'roc_auc':roc,'mcc':matthews_corrcoef(yt,yp),'pr_auc':prauc,'fpr_at_tpr99':fpr99,
            'benign_f1':cr['Benign']['f1-score'],'attack_f1':cr['Attack']['f1-score'],
            'fpr_at_tpr95':fpr95,
            'loss':tl/max(len(dl),1),'predictions':yp,'labels':yt,'probabilities':ypr}

def _selection_score(m):
    """Early-stopping criterion on the VALIDATION set: F1 when the validation
       windows contain both classes, otherwise negative validation loss."""
    return m['f1'] if len(np.unique(m['labels'])) > 1 else -m['loss']

def train_full(mdl, trdl, vadl, crit, epochs, lr, wd, es, wu=5,
               is_v2=False, aw=0.1, verbose=True, coral_wt=None, tedl=None):
    """Train with early stopping and best-checkpoint selection on `vadl`, the
       VALIDATION loader (never the test set). After the best checkpoint is
       restored, the returned metrics are computed on `tedl` (test) if given,
       otherwise on `vadl`. `hist` holds per-epoch VALIDATION metrics.
       REVISION (R1-m1): selection previously monitored the test loader."""
    if coral_wt is None: coral_wt = getattr(Config, 'CORAL_WT', 0.0)
    opt = optim.AdamW(mdl.parameters(), lr=lr, weight_decay=wd)
    sch = get_sched(opt, wu, epochs)
    scaler = _make_scaler()
    bf=-np.inf; bs_=None; pat=0; hist=[]
    ep_iter = tqdm(range(epochs), desc="Epochs", leave=True) if verbose else range(epochs)
    for ep in ep_iter:
        if is_v2: tl,_ = train_ep_v2(mdl,trdl,crit,opt,aw,scaler,coral_wt=coral_wt)
        else:     tl,_ = train_ep_base(mdl,trdl,crit,opt,scaler)
        sch.step(); m = evaluate(mdl,vadl,crit,is_v2, verbose=verbose)
        m['train_loss']=tl
        # Strip large per-epoch arrays — only needed in final evaluate call
        _m_hist = {k:v for k,v in m.items() if k not in ('predictions','labels','probabilities')}
        hist.append(_m_hist)
        _score = _selection_score(m)
        is_best = _score > bf
        if verbose:
            # BUG FIX: avoid nested quotes in f-strings (SyntaxError in Python <3.12)
            _f1=m['f1']; _auc=m['roc_auc']; _mcc=m['mcc']
            _bfv=m['benign_f1']; _afv=m['attack_f1']
            btag = " \u2605" if is_best else ""
            if hasattr(ep_iter,'set_postfix'):
                ep_iter.set_postfix(
                    loss="{:.4f}".format(tl), val_f1="{:.4f}".format(_f1),
                    auc="{:.4f}".format(_auc), mcc="{:.4f}".format(_mcc),
                    best="{:.4f}".format(bf))
            tqdm.write("  Ep {:3d}/{:d}  Loss={:.4f}  valF1={:.4f}  valAUC={:.4f}  valMCC={:.4f}"
                       "  BenF1={:.4f}  AtkF1={:.4f}{}".format(
                           ep+1, epochs, tl, _f1, _auc, _mcc, _bfv, _afv, btag))
        if is_best: bf=_score; bs_=copy.deepcopy(mdl.state_dict()); pat=0
        else:
            pat += 1
            if pat >= es:
                if verbose: tqdm.write("  \u23f9 Early stop at epoch {:d}  |  Best val score={:.4f}".format(ep+1,bf))
                break
    if bs_: mdl.load_state_dict(bs_)
    return evaluate(mdl, tedl if tedl is not None else vadl, crit, is_v2), hist

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def multi_seed(model_fn, seeds, epochs, is_v2=False, aw=0.1, wu=None,
               save_sd0=False, coral_wt=None,
               Xtr=None, ytr=None, ctr=None, Xte=None, yte=None, cte=None,
               Xva=None, yva=None, cva=None):
    if Xtr is None: Xtr,ytr,ctr = X_train,y_train,ctx_train
    if Xte is None: Xte,yte,cte = X_test,y_test,ctx_test
    if Xva is None: Xva,yva,cva = X_val,y_val,ctx_val
    all_m = []; all_h = []; _sd0 = None
    for i,s in enumerate(seeds):
        print("  -- Seed {} [{}/{}] --".format(s, i+1, len(seeds)))
        set_seed(s); mdl = model_fn().to(device)
        tl,tel = make_loaders(Xtr,ytr,ctr,Xte,yte,cte)
        vl = make_eval_loader(Xva,yva,cva)
        cr = make_criterion(ytr)
        _wu = wu if wu is not None else Config.WARMUP
        # Select on validation (vl); report on test (tel) — REVISION (R1-m1)
        m,h_ = train_full(mdl,tl,vl,cr,epochs,Config.LR,Config.WD,Config.EARLY_STOP,
                          _wu,is_v2,aw,coral_wt=coral_wt,tedl=tel)
        m['seed'] = s   # REVISION (R1-M1): explicit seed tag for seed-matched pairing
        # REVISION 2: keep validation scores for validation-selected operating points
        _mv = evaluate(mdl, vl, cr, is_v2, verbose=False)
        m['val_probabilities'] = _mv['probabilities']; m['val_labels'] = _mv['labels']
        print("  |  F1={:.4f}  AUC={:.4f}  MCC={:.4f}  PR-AUC={:.4f}".format(
              m['f1'],m['roc_auc'],m['mcc'],m['pr_auc']))
        print("  |  Acc={:.4f}  Prec={:.4f}  Rec={:.4f}".format(
              m['accuracy'],m['precision'],m['recall']))
        print("  |  Benign-F1={:.4f}  Attack-F1={:.4f}  FPR@TPR99={:.4f}  FPR@TPR95={:.4f}".format(
              m['benign_f1'],m['attack_f1'],m['fpr_at_tpr99'],m.get('fpr_at_tpr95',1.)))
        if save_sd0 and i == 0:
            _sd0 = copy.deepcopy(mdl.state_dict())  # seed-42 best weights for analysis
        all_m.append(m); all_h.append(h_); del mdl; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return all_m, all_h, _sd0

def summarise(ml):
    ks=['accuracy','precision','recall','f1','roc_auc','mcc','pr_auc','benign_f1','attack_f1','fpr_at_tpr99','fpr_at_tpr95','macro_f1','tpr','fpr']
    s={}
    for k in ks:
        v=[m[k] for m in ml if k in m]
        # REVISION (R1-M1): sample SD (ddof=1); previously population SD (ddof=0)
        if v: s[k+'_mean']=float(np.mean(v)); s[k+'_std']=float(np.std(v, ddof=1)) if len(v)>1 else 0.; s[k+'_vals']=v
    s['seeds'] = [m.get('seed') for m in ml]   # aligned with every *_vals list
    # Store concatenated probabilities and labels across all seeds for
    # full operational metric computation (Precision@TPR, TPR@FPR, Brier)
    all_probs  = [m['probabilities'] for m in ml if 'probabilities' in m and len(m['probabilities'])>0]
    all_labels = [m['labels']        for m in ml if 'labels'        in m and len(m['labels'])>0]
    if all_probs and all_labels:
        s['_probs_all']  = np.concatenate(all_probs)
        s['_labels_all'] = np.concatenate(all_labels)
    return s

# ── CORAL: source-domain second-order feature alignment ──────────────────────
def coral_loss(feats, domains):
    """Mean pairwise CORAL distance between per-domain covariances of `feats`.
       feats: (B,d) fused features; domains: (B,) integer dataset ids.
       Returns 0 if <2 domains (or <2 samples/domain) are present in the batch."""
    feats = feats.float()
    uniq = torch.unique(domains)
    if uniq.numel() < 2: return feats.new_zeros(())
    d = feats.size(1); covs = []
    for u in uniq:
        f = feats[domains == u]
        if f.size(0) < 2: continue
        f = f - f.mean(0, keepdim=True)
        covs.append((f.t() @ f) / (f.size(0) - 1))
    if len(covs) < 2: return feats.new_zeros(())
    loss = feats.new_zeros(()); cnt = 0
    for i in range(len(covs)):
        for j in range(i+1, len(covs)):
            loss = loss + ((covs[i] - covs[j])**2).sum() / (4.0 * d * d); cnt += 1
    return loss / max(cnt, 1)

# ── Transductive test-time BatchNorm adaptation (AdaBN, Li et al. 2017) ───────
@torch.no_grad()
def adapt_bn(mdl, loader, max_batches=None):
    """Recompute BN running stats from the (UNLABELED) target-domain data in
       `loader`, then return the model in eval mode. No labels are used. No-op
       for BN-free models (e.g. Transformer-IDS, LayerNorm-only)."""
    bns = [m for m in mdl.modules() if isinstance(m, nn.BatchNorm1d)]
    if not bns:
        mdl.eval(); return mdl
    was_training = mdl.training
    saved = [m.momentum for m in bns]
    for m in bns:
        m.reset_running_stats(); m.momentum = None; m.train()   # cumulative average
    for i, b in enumerate(loader):
        s = b['sequence'].to(device); ctx = b['context'].to(device)
        with torch.amp.autocast(_amp_dev, enabled=_amp_en):
            mdl(s, ctx)
        if max_batches and (i + 1) >= max_batches: break
    for m, mom in zip(bns, saved): m.momentum = mom; m.eval()
    mdl.eval()
    return mdl

# ── Threshold helpers (data-driven grid, percentile candidates) ──────────────
def best_threshold(probs, labels, n=200):
    cand = np.unique(np.percentile(probs, np.linspace(0, 100, n)))
    bf, bt = 0.0, 0.5
    for t in cand:
        f = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f > bf: bf, bt = f, t
    return bf, bt

def calibrated_f1(probs, labels, seed, cal_frac=0.20):
    """Threshold picked on a disjoint cal slice, F1 reported on the eval slice."""
    if probs is None or labels is None or len(probs) < 20:
        return (float(f1_score(labels, (probs>=0.5).astype(int), zero_division=0))
                if probs is not None and len(probs) else 0.0), 0.5, 0
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(probs)); n_cal = max(10, int(cal_frac*len(probs)))
    ci, ei = perm[:n_cal], perm[n_cal:]
    _, thr = best_threshold(probs[ci], labels[ci])
    return float(f1_score(labels[ei], (probs[ei] >= thr).astype(int), zero_division=0)), thr, n_cal

print("Training infrastructure ready  |  CORAL + test-time BN adaptation enabled")

Training infrastructure ready  |  CORAL + test-time BN adaptation enabled


## Revision 2 helpers: operating points at deployment priors, purged calibration, saving

In [14]:
def _thr_for_tpr(p, y, target):
    """Largest threshold whose TPR on (p, y) is >= target."""
    pos = np.sort(p[y == 1])[::-1]
    if len(pos) == 0: return 0.5
    k = int(np.ceil(target * len(pos))) - 1
    return float(pos[min(max(k, 0), len(pos) - 1)])

def ops_at_priors(val_p, val_y, test_p, test_y, priors=None, target_tpr=0.95):
    """REVISION 2 (R3 §3.6). Threshold chosen on VALIDATION for TPR >= target; TPR and
       FPR measured on TEST; precision and F1 at a deployment prior pi computed from the
       class-conditional rates: prec = pi*TPR / (pi*TPR + (1-pi)*FPR). This is what the
       detector would achieve on a test population with attack prevalence pi."""
    priors = priors or Config.OPS_PRIORS
    t = _thr_for_tpr(val_p, val_y, target_tpr)
    yp = (test_p >= t)
    tpr = float(yp[test_y == 1].mean()) if (test_y == 1).any() else 0.0
    fpr = float(yp[test_y == 0].mean()) if (test_y == 0).any() else 0.0
    out = {'threshold': t, 'tpr': tpr, 'fpr': fpr}
    for pi in priors:
        den = pi * tpr + (1 - pi) * fpr
        prec = pi * tpr / den if den > 0 else 0.0
        f1 = 2 * prec * tpr / (prec + tpr) if (prec + tpr) > 0 else 0.0
        out[f'precision@{pi}'] = prec; out[f'f1@{pi}'] = f1
    return out

def cal_eval_split(n, cal_frac=0.20, purge=1):
    """REVISION 2 (R3 §3.3). Target windows are non-overlapping and kept in row order,
       so the first cal_frac of windows covers the first cal_frac of target RECORDS.
       A purge gap of `purge` windows separates the slices. No record is shared."""
    n_cal = max(10, int(cal_frac * n))
    return np.arange(0, n_cal), np.arange(min(n_cal + purge, n), n)

def f1_at(p, y, t): return float(f1_score(y, (p >= t).astype(int), zero_division=0))

def _auc(p, y):
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

RESULTS = {'part': Config.PART, 'data_report': DATA_REPORT}
def save_results(tag=''):
    def _clean(o):
        if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
        if isinstance(o, np.ndarray): return _clean(o.tolist())
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        return o
    p = os.path.join(Config.OUT, f'part{Config.PART}_results.json')
    with open(p, 'w') as f: json.dump(_clean(RESULTS), f)
    print(f'  [saved {p}{" — " + tag if tag else ""}]  elapsed {(time.time()-RUN_T0)/3600:.2f} h')

def time_left_ok(est_h=0.0):
    el = (time.time() - RUN_T0) / 3600
    ok = el + est_h < Config.TIME_BUDGET_H
    if not ok: print(f'  SKIPPED: elapsed {el:.2f} h + est {est_h:.2f} h exceeds budget {Config.TIME_BUDGET_H} h')
    return ok

def _strip(m):
    """Per-seed record without large arrays (arrays are saved separately)."""
    return {k: v for k, v in m.items() if not isinstance(v, np.ndarray)}

save_results('setup')

  [saved tch_net_v2_results/partC1_results.json — setup]  elapsed 0.01 h


## Shared LODO helpers (fold builder, one training run, scoring on the evaluation slice, budget curve)

In [15]:
def build_lodo_fold(held_name, feat_idx=None, add_mask=False, verbose=True):
    """REVISION 2. Per-dataset class balance, source train/val split (90/10 by position),
       scaler fit on source train rows. Source train windows overlap (stride S); source
       validation and all target windows are non-overlapping and kept in row order.
       feat_idx: optional subset of canonical features (common-feature control).
       add_mask: append the dataset's 46-d availability mask to every record (mask-aware)."""
    vf = Config.LODO_VAL_FRAC
    tr, va, te = ([], [], [], []), ([], [], [], []), ([], [], [], [])
    for di, (dsn, dsd) in enumerate(loader.datasets.items()):
        Xd = dsd['X'].copy(); yd = dsd['y'].copy(); cd = dsd['ctx'].copy()
        np.nan_to_num(Xd, copy=False, nan=0., posinf=1e6, neginf=-1e6)
        rng_l = np.random.RandomState(Config.DATA_SEED)
        ben = np.where(yd == 0)[0]; atk = np.where(yd == 1)[0]
        if len(ben) == 0 or len(atk) == 0: continue
        n_min = min(len(ben), len(atk))
        keep = np.sort(np.concatenate([rng_l.choice(ben, n_min, replace=False), rng_l.choice(atk, n_min, replace=False)]))
        Xd, yd, cd = Xd[keep], yd[keep], cd[keep]
        if dsn == held_name:
            te[0].append(Xd); te[1].append(yd); te[2].append(cd); te[3].append(np.full(len(yd), di, np.int32))
        else:
            n_va = int(len(yd) * vf); n_tr = len(yd) - n_va
            for part, sl in ((tr, slice(0, n_tr)), (va, slice(n_tr, None))):
                part[0].append(Xd[sl]); part[1].append(yd[sl]); part[2].append(cd[sl]); part[3].append(np.full(len(yd[sl]), di, np.int32))
    if not tr[0] or not te[0]: return None
    st = lambda p: (np.vstack(p[0]).astype(np.float32), np.hstack(p[1]).astype(np.int32), np.vstack(p[2]).astype(np.int32), np.hstack(p[3]))
    (Xtr, ytr, ctr, dtr), (Xva, yva, cva, dva), (Xte, yte, cte, dte) = st(tr), st(va), st(te)
    sc = RobustScaler(quantile_range=(5, 95))
    Xtr = np.clip(sc.fit_transform(Xtr), -10, 10).astype(np.float32)
    Xva = np.clip(sc.transform(Xva), -10, 10).astype(np.float32)
    Xte = np.clip(sc.transform(Xte), -10, 10).astype(np.float32)
    if feat_idx is not None:
        Xtr, Xva, Xte = Xtr[:, feat_idx], Xva[:, feat_idx], Xte[:, feat_idx]
    if add_mask:
        _mk = np.stack([[1.0 if f in COVERED[n] else 0.0 for f in SEMANTIC_FEATURES] for n in DS_NAMES]).astype(np.float32)
        Xtr = np.hstack([Xtr, _mk[dtr]]); Xva = np.hstack([Xva, _mk[dva]]); Xte = np.hstack([Xte, _mk[dte]])
    W = Config.WINDOW_SIZE
    trs = create_sequences(Xtr, ytr, ctr, dtr, W, Config.STRIDE)
    vas = create_sequences(Xva, yva, cva, dva, W, Config.EVAL_STRIDE)
    tes = create_sequences(Xte, yte, cte, np.zeros(len(yte), np.int32), W, Config.EVAL_STRIDE)  # row order kept
    trs = _cap(*trs, Config.MAX_TRAIN_SEQ, f'ltr_{held_name[:6]}')
    vas = _cap(*vas, Config.MAX_VAL_SEQ, f'lva_{held_name[:6]}')
    if len(np.unique(tes[1])) < 2: return None
    return {'train': trs[:3], 'val': vas[:3], 'test': tes[:3], 'nf': Xtr.shape[1],
            'atk_frac': float(tes[1].mean()), 'n_target_records': int(len(yte))}

def lodo_run(model_fn, is_v2, fold, seed, coral_wt=None, ttbn=True):
    """One LODO training run; every metric on the same evaluation slice."""
    (Xtr, ytr, ctr), (Xva, yva, cva), (Xte, yte, cte) = fold['train'], fold['val'], fold['test']
    set_seed(seed); mdl = model_fn().to(device)
    tl, tel = make_loaders(Xtr, ytr, ctr, Xte, yte, cte); vl = make_eval_loader(Xva, yva, cva)
    cr = make_criterion(ytr)
    m_raw, _ = train_full(mdl, tl, vl, cr, Config.LODO_EPOCHS, Config.LR, Config.WD, es=5, wu=Config.WARMUP_FAST,
                          is_v2=is_v2, aw=Config.AUX_WT, verbose=False, coral_wt=coral_wt, tedl=tel)
    p_raw = np.asarray(m_raw['probabilities']); y = np.asarray(m_raw['labels'])
    n_bn = sum(isinstance(m, nn.BatchNorm1d) for m in mdl.modules())
    if ttbn and n_bn > 0:
        adapt_bn(mdl, tel); p_ad = np.asarray(evaluate(mdl, tel, cr, is_v2=is_v2, verbose=False)['probabilities'])
    else:
        p_ad = p_raw.copy()
    del mdl; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return p_raw, p_ad, y, n_bn

def score_run(p_raw, p_ad, y):
    ci, ei = cal_eval_split(len(y))
    P = float(y[ei].mean())
    t_raw = best_threshold(p_raw[ci], y[ci])[1]; t_ad = best_threshold(p_ad[ci], y[ci])[1]
    return {'n_cal': len(ci), 'n_eval': len(ei), 'eval_attack_frac': P,
            'trivial_all_attack_f1': 2 * P / (1 + P), 'trivial_all_benign_macro_f1': (1 - P) / (2 - P) if P < 1 else 0.0,
            'f1_raw': f1_at(p_raw[ei], y[ei], .5), 'f1_ad': f1_at(p_ad[ei], y[ei], .5),
            'f1_cal': f1_at(p_raw[ei], y[ei], t_raw), 'f1_ad_cal': f1_at(p_ad[ei], y[ei], t_ad),
            'f1_oracle_ad': best_threshold(p_ad[ei], y[ei])[0],
            'macro_f1_ad_cal': float(f1_score(y[ei], (p_ad[ei] >= t_ad).astype(int), average='macro', zero_division=0)),
            'auc_raw': _auc(p_raw[ei], y[ei]), 'auc_ad': _auc(p_ad[ei], y[ei]), 'thr_ad_cal': t_ad}

def budget_curve(p, y, ks=(8, 16, 32, 64, 128, 256), draws=50, seed=0):
    """REVISION 2 (R3 §3.10). Calibrate on k labelled windows drawn from the calibration
       slice (k*W labelled records); report F1 on the evaluation slice."""
    ci, ei = cal_eval_split(len(y)); r = np.random.RandomState(seed); out = {}
    for k in ks:
        if k > len(ci): continue
        f = []
        for _ in range(draws):
            s = r.choice(ci, k, replace=False)
            if len(np.unique(y[s])) < 2: f.append(f1_at(p[ei], y[ei], 0.5)); continue
            f.append(f1_at(p[ei], y[ei], best_threshold(p[s], y[s])[1]))
        out[k] = {'mean': float(np.mean(f)), 'p10': float(np.percentile(f, 10)), 'p90': float(np.percentile(f, 90)),
                  'labelled_records': int(k * Config.WINDOW_SIZE)}
    return out

# Part C1 — LODO: every model, 5 seeds, purged calibration, labelling-budget curve
For each held-out dataset, models train on the other three (early stopping on source validation).
All numbers for a run are computed on the SAME evaluation slice of the target: the first 20% of
target records (non-overlapping windows) is the calibration slice, the rest is evaluation (R3 §3.3).

In [16]:
LODO_MODELS = {'TCH-Net': (make_tch_v2, True),
               'H-only': ((lambda: TCHNetV2(N_FEATURES, Config.WINDOW_SIZE, nc=Config.N_CLASSES, use_t=False)), True)}
LODO_MODELS.update({b: (make_baseline(b), False) for b in Config.LODO_BASELINE_NAMES})
RESULTS['C1_runs'] = []; RESULTS['C1_budget'] = []; RESULTS['C1_folds'] = {}
PROBS_OUT = os.path.join(Config.OUT, 'lodo_probs'); os.makedirs(PROBS_OUT, exist_ok=True)
for held in DS_NAMES:
    fold = build_lodo_fold(held)
    if fold is None: continue
    RESULTS['C1_folds'][held] = {'n_target_windows': int(len(fold['test'][1])), 'atk_frac': fold['atk_frac'],
                                 'n_target_records': fold['n_target_records']}
    print(f"\n{'─'*60}\n  held-out {held}: target windows {len(fold['test'][1]):,} (attack {fold['atk_frac']*100:.1f}%)")
    for name, (fn, v2) in LODO_MODELS.items():
        for seed in Config.LODO_SEEDS:
            if not time_left_ok(0.1): break
            p_raw, p_ad, y, n_bn = lodo_run(fn, v2, fold, seed)
            r = {'model': name, 'held_out': held, 'seed': seed, 'bn_layers': n_bn, **score_run(p_raw, p_ad, y)}
            RESULTS['C1_runs'].append(r)
            RESULTS['C1_budget'].append({'model': name, 'held_out': held, 'seed': seed, 'curve': budget_curve(p_ad, y)})
            np.savez_compressed(os.path.join(PROBS_OUT, f'{name}_{held[:12]}_{seed}.npz'),
                                p_raw=p_raw.astype(np.float16), p_ad=p_ad.astype(np.float16), y=y.astype(np.int8))
            print(f"    {name:<16} s{seed}: raw={r['f1_raw']:.3f} ad={r['f1_ad']:.3f} cal={r['f1_cal']:.3f} "
                  f"ad+cal={r['f1_ad_cal']:.3f} auc_ad={r['auc_ad']:.3f} (trivial {r['trivial_all_attack_f1']:.3f})")
        save_results(f'{held} {name}')
    del fold; gc.collect()

  [ltr_CICIDS] 99,861 seqs (ben=50,019 50.1% | atk=49,842 49.9%)
  [lva_CICIDS] 1,344 seqs (ben=661 49.2% | atk=683 50.8%)

────────────────────────────────────────────────────────────
  held-out CICIDS-2017: target windows 5,146 (attack 49.4%)


  batches:   0%|          | 0/195 [00:00<?, ?it/s]

  batches:   0%|          | 0/195 [00:00<?, ?it/s]

  batches:   0%|          | 0/195 [00:00<?, ?it/s]

  batches:   0%|          | 0/195 [00:00<?, ?it/s]

...

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  batches:   0%|          | 0/221 [00:00<?, ?it/s]

  eval:   0%|          | 0/7 [00:00<?, ?it/s]

    1D-CNN-IDS       s2024: raw=0.000 ad=0.528 cal=0.667 ad+cal=0.668 auc_ad=0.521 (trivial 0.668)
  [saved tch_net_v2_results/partC1_results.json — UNSW-NB15 1D-CNN-IDS]  elapsed 9.12 h


In [17]:
print('=' * 70 + '\nLODO: classical models on the same windows (flattened), calibrated\n' + '=' * 70)
# REVISION 2 (R3 §3.10): RF and XGBoost now see the same windows and the same evaluation
# slice as the neural models, and get the same threshold calibration (no BN, so no TTBN).
for held in DS_NAMES:
    fold = build_lodo_fold(held, verbose=False)
    if fold is None: continue
    Xtr = fold['train'][0].reshape(len(fold['train'][0]), -1); ytr = fold['train'][1]
    Xte = fold['test'][0].reshape(len(fold['test'][0]), -1); yte = fold['test'][1]
    for cn in ['Random-Forest', 'XGBoost']:
        for seed in Config.LODO_SEEDS:
            r_ = np.random.RandomState(seed); idx = r_.choice(len(Xtr), min(100_000, len(Xtr)), replace=False)
            clf = _clf(cn, seed) if '_clf' in dir() else None
            if clf is None:
                clf = (RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=seed) if cn == 'Random-Forest' or not XGB_OK
                       else XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1, random_state=seed, verbosity=0, eval_metric='logloss'))
            clf.fit(Xtr[idx], ytr[idx]); p = clf.predict_proba(Xte)[:, 1]
            r = {'model': cn, 'held_out': held, 'seed': seed, 'bn_layers': 0, **score_run(p, p, yte)}
            RESULTS['C1_runs'].append(r)
            RESULTS['C1_budget'].append({'model': cn, 'held_out': held, 'seed': seed, 'curve': budget_curve(p, yte)})
            print(f"    {cn:<14} {held[:14]:<14} s{seed}: raw={r['f1_raw']:.3f} cal={r['f1_cal']:.3f} auc={r['auc_raw']:.3f}")
    save_results(f'classical {held}')
    del fold; gc.collect()

LODO: classical models on the same windows (flattened), calibrated


  [ltr_CICIDS] 99,861 seqs (ben=50,019 50.1% | atk=49,842 49.9%)
  [lva_CICIDS] 1,344 seqs (ben=661 49.2% | atk=683 50.8%)


    Random-Forest  CICIDS-2017    s42: raw=0.430 cal=0.657 auc=0.574


    Random-Forest  CICIDS-2017    s123: raw=0.482 cal=0.660 auc=0.568


    Random-Forest  CICIDS-2017    s456: raw=0.496 cal=0.661 auc=0.559


...

    Random-Forest  CICIoMT-2024   s123: raw=0.402 cal=0.671 auc=0.524


    Random-Forest  CICIoMT-2024   s456: raw=0.613 cal=0.671 auc=0.554


    Random-Forest  CICIoMT-2024   s789: raw=0.488 cal=0.671 auc=0.451


    Random-Forest  CICIoMT-2024   s2024: raw=0.671 cal=0.671 auc=0.497


    XGBoost        CICIoMT-2024   s42: raw=0.671 cal=0.671 auc=0.447


In [ ]:
_df = pd.DataFrame(RESULTS['C1_runs'])
print('\nMEAN OVER FOLDS x SEEDS (all on the evaluation slice)')
print(_df.groupby('model')[['f1_raw', 'f1_ad', 'f1_cal', 'f1_ad_cal', 'f1_oracle_ad', 'auc_ad', 'trivial_all_attack_f1']].mean().round(4).to_string())
print('\nPER FOLD (adapted + calibrated F1)')
print(_df.pivot_table(index='model', columns='held_out', values='f1_ad_cal', aggfunc='mean').round(4).to_string())
_df['restored'] = _df['auc_ad'] > 0.60
print('\nRESTORED RANKING (post-adaptation AUC > 0.60): share of runs and mean calibrated F1')
print(_df.groupby(['model', 'restored'])['f1_ad_cal'].agg(['count', 'mean']).round(4).to_string())
save_results('C1 summary')

## Export
Download the whole `tch_net_v2_results` folder from the Output tab (it contains `part<X>_results.json`).

In [ ]:
save_results('final')
print(f'Total elapsed: {(time.time() - RUN_T0) / 3600:.2f} h')
print(sorted(os.listdir(Config.OUT)))